In [1]:
# EquiBind Batch Docking Pipeline with Multiple Pose Generation
#
# This notebook docks all protein-ligand combinations using EquiBind,
# generating multiple diverse poses per combination using RDKit conformer generation.
#
# Strategy: Generate multiple 3D conformers with RDKit's EmbedMultipleConfs(),
# then run EquiBind on each conformer to produce diverse docked poses.
#
# Uses conda environment 'equibind' to run multiligand_inference.py:
#   conda run -n equibind python ~/docking_tools/EquiBind/multiligand_inference.py \
#     -o ./equibind_out -r protein.pdb -l ligand.sdf --device cuda

# Imports

In [2]:
from __future__ import annotations

import csv
import hashlib
import itertools
import json
import os
import shutil
import subprocess
import threading
import time
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
from datetime import datetime
from enum import IntEnum
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Descriptors, Lipinski, rdMolAlign, rdMolDescriptors, rdForceFieldHelpers
from rdkit.Chem import rdMolTransforms
from rdkit.ForceField import rdForceField

# Suppress RDKit deprecation warnings (e.g. GetValence)
RDLogger.DisableLog('rdApp.*')

# Config

In [3]:

# ── Configuration ──────────────────────────────────────────────────────────

# Paths
workspace_root = Path.cwd()
drugs_dir = Path("storage/ligands_sdf_large_approved")
receptors_dir = workspace_root / "Orai"
RECEPTOR_NAME_FILTER: str = "_cleaned"  # Only use receptor files containing this substring (empty = all)

# Pocket result directories (from previous runs)
fpocket_results_folder = Path("/workspace/storage/pocket_results/fpocket_results")
p2rank_folder = Path("/workspace/storage/pocket_results/p2rank_results")

# EquiBind paths
EQUIBIND_DIR = Path("/workspace/EquiBind")
EQUIBIND_MULTILIGAND_SCRIPT = EQUIBIND_DIR / "multiligand_inference.py"
EQUIBIND_DEVICE = "cuda"  # "cpu" or "cuda"

# Output directories
POCKET_GUIDED_OUTPUT_DIR = workspace_root / "equibind_pocket_guided"
POCKET_GUIDED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EQUIBIND_BATCH_DIR = workspace_root / "equibind_batches"
EQUIBIND_OUTPUT_DIR = workspace_root / "equibind_docked_poses"
EQUIBIND_BATCH_DIR.mkdir(parents=True, exist_ok=True)
EQUIBIND_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LOG_DIR = POCKET_GUIDED_OUTPUT_DIR / "wdlogs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Pocket-guided docking settings
N_TOP_POCKETS: int = 5          # Top-ranked pockets to use from each method
POSES_PER_POCKET: int = 3       # Poses to generate per pocket (using different conformers)
N_UNGUIDED_POSES: int = 10      # Natural EquiBind poses per protein-ligand pair
POCKET_MATCH_THRESHOLD: float = 8.0  # Distance (Å) to consider a pose "inside" a pocket

# Pocket enforcement settings
FORCE_POCKET: bool = True        # If True, reject poses whose centroid drifts outside the pocket
MAX_POCKET_TRIALS: int = 200      # Max conformer trials per desired pose when FORCE_POCKET is True
CLAMP_POSE_TO_POCKET: bool = True  # If True, clamp drifted poses back to pocket (instead of rejecting)

# Protein cropping settings (constrains EquiBind's search space to the pocket region)
USE_PROTEIN_CROPPING: bool = True   # If True, dock against cropped protein (pocket only)
POCKET_CROP_RADIUS: float = 6.0    # Residues within this radius of pocket center are kept
POCKET_CROP_BUFFER: float = 3.0     # Additional buffer for context

# Batch docking settings
NUM_POSES: int = 30             # Unique poses per protein-ligand combination
NUM_CONFORMERS: int = 10        # RDKit conformers to generate (>= NUM_POSES for diversity)
MAX_ATTEMPTS: int = NUM_POSES * 30  # Fallback if conformers fail
POSE_RMSD_THRESHOLD: float = 1.0    # Min RMSD (Å) between poses to consider distinct

# RDKit conformer generation parameters
RDKIT_SEEDS: list = [42, 123, 456, 789, 1001, 2022, 3141, 5926, 8675, 9999]
CONFORMERS_PER_SEED: int = 5
RDKIT_RANDOM_SEED: int = 42
RDKIT_NUM_THREADS: int = 0              # 0 = use all available threads
RDKIT_PRUNE_RMS_THRESH: float = 0.2     # Prune similar conformers during generation

# UFF post-docking minimization settings
UFF_MINIMIZE: bool = True            # Enable UFF minimization of docked poses
UFF_MAX_ITERS: int = 500             # Maximum minimization iterations
UFF_ENERGY_TOL: float = 1e-4         # Energy convergence tolerance (kcal/mol)
UFF_FORCE_TOL: float = 1e-3          # Force convergence tolerance
UFF_PROTEIN_CONSTRAINT_WT: float = 100.0  # Constraint weight for protein atoms (kcal/mol/Å²)
UFF_PROXIMITY_RADIUS: float = 8.0    # Only include protein residues within this Å of ligand
UFF_ADD_HYDROGENS: bool = True       # Add hydrogens before minimization for better UFF terms
UFF_VDW_THRESH: float = 0.1          # Non-bonded interaction cutoff threshold

# Parallelism settings
N_PARALLEL_WORKERS: int = min(os.cpu_count() or 4, 8)  # Max threads for parallel tasks
PARALLEL_CONFORMER_GEN: bool = True   # Parallelize conformer generation across seeds
PARALLEL_POCKETS: bool = True         # Process pockets concurrently (GPU calls are serialized)
PARALLEL_UFF: bool = True             # Parallelize UFF minimization of docked poses

# Overwrite / skip settings
SKIP_EXISTING: bool = False
OVERWRITE_EXISTING: bool = True  # Set to True to re-dock existing combinations

# Error log filename (one per output directory)
ERROR_LOG_FILENAME = "failed_docking.json"

# Check for reduce executable (for adding hydrogens to proteins)
REDUCE_EXECUTABLE = shutil.which("reduce")

# ── Validate ───────────────────────────────────────────────────────────────
if not drugs_dir.exists():
    raise FileNotFoundError(f"Ligand directory missing: {drugs_dir}")
if not receptors_dir.exists():
    raise FileNotFoundError(f"Receptor directory missing: {receptors_dir}")

print("=" * 80)
print("EquiBind Batch Docking Configuration")
print("=" * 80)
print(f"Number of poses per combination: {NUM_POSES}")
print(f"RDKit conformers to generate:    {NUM_CONFORMERS}")
print(f"Maximum attempts per combination:{MAX_ATTEMPTS}")
print(f"Pose RMSD threshold:             {POSE_RMSD_THRESHOLD} Å")
print(f"RDKit prune RMS threshold:       {RDKIT_PRUNE_RMS_THRESH} Å")
print(f"EquiBind script:  {EQUIBIND_MULTILIGAND_SCRIPT}")
print(f"EquiBind device:  {EQUIBIND_DEVICE}")
print(f"Batch directory:  {EQUIBIND_BATCH_DIR}")
print(f"Output directory: {EQUIBIND_OUTPUT_DIR}")
print()
print(f"UFF post-docking minimization:   {'ON' if UFF_MINIMIZE else 'OFF'}")
if UFF_MINIMIZE:
    print(f"  Max iterations:                {UFF_MAX_ITERS}")
    print(f"  Protein constraint weight:     {UFF_PROTEIN_CONSTRAINT_WT} kcal/mol/Å²")
    print(f"  Proximity radius:              {UFF_PROXIMITY_RADIUS} Å")
print()

if not EQUIBIND_DIR.exists():
    print(f"⚠️  WARNING: EquiBind directory not found at {EQUIBIND_DIR}")
    print("   Please update EQUIBIND_DIR to point to your EquiBind installation")
elif not EQUIBIND_MULTILIGAND_SCRIPT.exists():
    print(f"⚠️  WARNING: multiligand_inference.py not found at {EQUIBIND_MULTILIGAND_SCRIPT}")
else:
    print(f"✓ EquiBind script found")


EquiBind Batch Docking Configuration
Number of poses per combination: 30
RDKit conformers to generate:    10
Maximum attempts per combination:900
Pose RMSD threshold:             1.0 Å
RDKit prune RMS threshold:       0.2 Å
EquiBind script:  /workspace/EquiBind/multiligand_inference.py
EquiBind device:  cuda
Batch directory:  /home/master_dev/equibind_batches
Output directory: /home/master_dev/equibind_docked_poses

UFF post-docking minimization:   ON
  Max iterations:                500
  Protein constraint weight:     100.0 kcal/mol/Å²
  Proximity radius:              8.0 Å

✓ EquiBind script found


# Docking Log & Property Tracking

In [4]:
# ============================================================================
# DOCKING LOG & PROPERTY TRACKING
# ============================================================================
# Adapted from the DiffDock pipeline: CSV docking log with ligand/protein
# properties, per-combination timing, error logging, and GPU detection.
# ============================================================================

DOCKING_LOG_FILENAME = "docking_log.csv"

DOCKING_LOG_COLUMNS = [
    "timestamp", "combo_name", "protein_name", "ligand_name",
    "status", "error_reason", "elapsed_time_s",
    # Pose counts
    "total_poses", "fpocket_poses", "p2rank_poses", "unguided_poses", "failed_poses",
    # Ligand properties
    "lig_molecular_weight", "lig_heavy_atoms", "lig_total_atoms",
    "lig_rotatable_bonds", "lig_num_rings", "lig_aromatic_rings",
    "lig_hbd", "lig_hba", "lig_tpsa", "lig_logp", "lig_formula",
    # Protein properties
    "prot_num_residues", "prot_num_atoms", "prot_num_chains",
    # Config & hardware
    "n_top_pockets", "poses_per_pocket", "n_unguided_poses",
    "force_pocket", "uff_minimize", "device", "gpu_model",
]


def get_gpu_model() -> str:
    """Return the GPU model name via nvidia-smi, or 'N/A' if unavailable."""
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=gpu_name", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=5,
        )
        names = [line.strip() for line in result.stdout.strip().splitlines() if line.strip()]
        return names[0] if names else "N/A"
    except Exception:
        return "N/A"


def get_ligand_properties(sdf_path: Path) -> Dict:
    """Extract molecular properties from a ligand SDF file using RDKit."""
    props = {
        "lig_molecular_weight": None, "lig_heavy_atoms": None,
        "lig_total_atoms": None, "lig_rotatable_bonds": None,
        "lig_num_rings": None, "lig_aromatic_rings": None,
        "lig_hbd": None, "lig_hba": None, "lig_tpsa": None,
        "lig_logp": None, "lig_formula": None,
    }
    try:
        supplier = Chem.SDMolSupplier(str(sdf_path), sanitize=False, removeHs=False)
        mol = next(iter(supplier), None)
        if mol is None:
            # Try loading as PDB
            mol = Chem.MolFromPDBFile(str(sdf_path), removeHs=False, sanitize=False)
        if mol is None:
            return props
        try:
            Chem.SanitizeMol(mol)
        except Exception:
            pass

        props["lig_molecular_weight"] = round(Descriptors.MolWt(mol), 2)
        props["lig_heavy_atoms"] = mol.GetNumHeavyAtoms()
        props["lig_total_atoms"] = mol.GetNumAtoms()
        props["lig_rotatable_bonds"] = rdMolDescriptors.CalcNumRotatableBonds(mol)
        ri = mol.GetRingInfo()
        props["lig_num_rings"] = ri.NumRings()
        props["lig_aromatic_rings"] = rdMolDescriptors.CalcNumAromaticRings(mol)
        props["lig_hbd"] = Lipinski.NumHDonors(mol)
        props["lig_hba"] = Lipinski.NumHAcceptors(mol)
        props["lig_tpsa"] = round(Descriptors.TPSA(mol), 2)
        props["lig_logp"] = round(Descriptors.MolLogP(mol), 2)
        props["lig_formula"] = rdMolDescriptors.CalcMolFormula(mol)
    except Exception as e:
        print(f"  ⚠ Could not compute ligand properties for {sdf_path.name}: {e}")
    return props


def get_protein_properties(pdb_path: Path) -> Dict:
    """Extract basic protein properties by parsing PDB ATOM records."""
    props = {"prot_num_residues": None, "prot_num_atoms": None, "prot_num_chains": None}
    try:
        residues = set()
        chains = set()
        atom_count = 0
        with open(pdb_path, 'r') as f:
            for line in f:
                if line.startswith("ATOM"):
                    atom_count += 1
                    chain_id = line[21]
                    res_seq = line[22:27].strip()
                    residues.add((chain_id, res_seq))
                    chains.add(chain_id)
        props["prot_num_residues"] = len(residues)
        props["prot_num_atoms"] = atom_count
        props["prot_num_chains"] = len(chains)
    except Exception as e:
        print(f"  ⚠ Could not parse protein {pdb_path.name}: {e}")
    return props


def ligand_props_valid(props: Dict) -> bool:
    """Return True if at least one key ligand property was computed successfully."""
    return props.get("lig_heavy_atoms") is not None


def precompute_properties(
    proteins: List[Path],
    ligands: List[Path],
) -> Tuple[Dict[Path, Dict], Dict[Path, Dict]]:
    """Pre-compute molecular properties for all proteins and ligands before docking.

    Returns (lig_props_cache, prot_props_cache) keyed by file path.
    """
    lig_props_cache: Dict[Path, Dict] = {}
    prot_props_cache: Dict[Path, Dict] = {}

    print("Pre-computing ligand properties...")
    for lig in ligands:
        lig_props_cache[lig] = get_ligand_properties(lig)
    print(f"  {len(lig_props_cache)} ligands analysed")

    print("Pre-computing protein properties...")
    for prot in proteins:
        prot_props_cache[prot] = get_protein_properties(prot)
    print(f"  {len(prot_props_cache)} proteins analysed")

    return lig_props_cache, prot_props_cache


# Detect GPU once at definition time — reused for every log row
_GPU_MODEL = get_gpu_model()
print(f"GPU detected: {_GPU_MODEL}")


def _build_log_row(
    combo_name: str,
    protein_name: str,
    ligand_name: str,
    status: str,
    error_reason: str,
    elapsed_time: float,
    total_poses: int,
    fpocket_poses: int,
    p2rank_poses: int,
    unguided_poses: int,
    failed_poses: int,
    lig_props: Dict,
    prot_props: Dict,
) -> Dict:
    """Build a single row dict for the docking log CSV."""
    row = {
        "timestamp": datetime.now().isoformat(),
        "combo_name": combo_name,
        "protein_name": protein_name,
        "ligand_name": ligand_name,
        "status": status,
        "error_reason": error_reason,
        "elapsed_time_s": round(elapsed_time, 2),
        "total_poses": total_poses,
        "fpocket_poses": fpocket_poses,
        "p2rank_poses": p2rank_poses,
        "unguided_poses": unguided_poses,
        "failed_poses": failed_poses,
        "n_top_pockets": N_TOP_POCKETS,
        "poses_per_pocket": POSES_PER_POCKET,
        "n_unguided_poses": N_UNGUIDED_POSES,
        "force_pocket": FORCE_POCKET,
        "uff_minimize": UFF_MINIMIZE,
        "device": EQUIBIND_DEVICE,
        "gpu_model": _GPU_MODEL,
    }
    row.update(lig_props)
    row.update(prot_props)
    return row


def init_docking_log(output_dir: Path) -> Tuple[Path, set]:
    """Initialise the docking log CSV and return (log_path, existing_combo_names).

    Creates the file with a header row if it does not yet exist.
    """
    log_path = output_dir / DOCKING_LOG_FILENAME
    existing_combos: set = set()

    if log_path.exists():
        try:
            df = pd.read_csv(log_path)
            existing_combos = set(df["combo_name"])
        except Exception:
            pass
    else:
        pd.DataFrame(columns=DOCKING_LOG_COLUMNS).to_csv(log_path, index=False)

    return log_path, existing_combos


def append_log_row(
    log_path: Path,
    existing_combos: set,
    combo_name: str,
    protein_name: str,
    ligand_name: str,
    status: str,
    error_reason: str,
    elapsed_time: float,
    combo_results: list,
    lig_props: Dict,
    prot_props: Dict,
) -> None:
    """Append a single combination's log row to the CSV, skipping duplicates."""
    if combo_name in existing_combos:
        return

    fpocket_poses = sum(1 for r in combo_results if r.mode == "fpocket" and r.success)
    p2rank_poses = sum(1 for r in combo_results if r.mode == "p2rank" and r.success)
    unguided_poses = sum(1 for r in combo_results if r.mode == "unguided" and r.success)
    failed_poses = sum(1 for r in combo_results if not r.success)
    total_poses = fpocket_poses + p2rank_poses + unguided_poses

    row = _build_log_row(
        combo_name=combo_name,
        protein_name=protein_name,
        ligand_name=ligand_name,
        status=status,
        error_reason=error_reason,
        elapsed_time=elapsed_time,
        total_poses=total_poses,
        fpocket_poses=fpocket_poses,
        p2rank_poses=p2rank_poses,
        unguided_poses=unguided_poses,
        failed_poses=failed_poses,
        lig_props=lig_props,
        prot_props=prot_props,
    )

    df = pd.DataFrame([row], columns=DOCKING_LOG_COLUMNS)
    df.to_csv(log_path, mode='a', header=False, index=False)
    existing_combos.add(combo_name)
    print(f"  📝 Logged {combo_name} to {log_path.name} ({len(existing_combos)} total)")


# ── Error Log Functions ────────────────────────────────────────────────────

def load_error_log(output_dir: Path) -> Dict[str, dict]:
    """Load the error log for a given output directory."""
    error_log_path = output_dir / ERROR_LOG_FILENAME
    if error_log_path.exists():
        try:
            with open(error_log_path, 'r') as f:
                return json.load(f)
        except (json.JSONDecodeError, Exception) as e:
            print(f"  ⚠ Could not read error log {error_log_path}: {e}")
    return {}


def save_error_log(output_dir: Path, error_log: Dict[str, dict]):
    """Save the error log for a given output directory."""
    error_log_path = output_dir / ERROR_LOG_FILENAME
    output_dir.mkdir(parents=True, exist_ok=True)
    with open(error_log_path, 'w') as f:
        json.dump(error_log, f, indent=2)


def record_failure(output_dir: Path, combo_name: str, error_message: str,
                   protein_name: str, ligand_name: str, elapsed_time: float):
    """Record a single failure to the error log."""
    error_log = load_error_log(output_dir)
    error_log[combo_name] = {
        "error": error_message,
        "timestamp": datetime.now().isoformat(),
        "protein": protein_name,
        "ligand": ligand_name,
        "elapsed_time": round(elapsed_time, 2),
    }
    save_error_log(output_dir, error_log)


print("✓ Docking log & property tracking functions loaded")
print(f"  Log filename: {DOCKING_LOG_FILENAME}")
print(f"  Error log filename: {ERROR_LOG_FILENAME}")
print(f"  Log columns: {len(DOCKING_LOG_COLUMNS)}")

GPU detected: NVIDIA A100-SXM4-80GB
✓ Docking log & property tracking functions loaded
  Log filename: docking_log.csv
  Error log filename: failed_docking.json
  Log columns: 33


# Signal Monitor

In [5]:
# ── Signal Monitor ──────────────────────────────────────────────────────────
# Three verbosity levels control what gets printed during docking.
#
#   MonitorLevel.ALL      → Everything: EquiBind invocations, returned poses,
#                           pocket distance checks, conformer generation, etc.
#   MonitorLevel.WARNING  → Rejected poses, dock failures, pocket drift,
#                           plus everything from CRITICAL.
#   MonitorLevel.CRITICAL → Only fatal errors (timeouts, missing output, total
#                           combo failures).
# ────────────────────────────────────────────────────────────────────────────


class MonitorLevel(IntEnum):
    """Signal monitoring verbosity levels."""
    ALL = 1          # Full trace
    WARNING = 2      # Warnings + critical only
    CRITICAL = 3     # Fatal errors only

# ── Set the active monitoring level here ──────────────────────────────────
# Change this to MonitorLevel.WARNING or MonitorLevel.CRITICAL to reduce output.
MONITOR_LEVEL = MonitorLevel.ALL


class DockingMonitor:
    """Structured signal monitor for the EquiBind docking pipeline.

    Every message is tagged with a level; only messages at or above the
    configured threshold are printed.  Colour prefixes make it easy to
    scan terminal output.
    """

    _PREFIXES = {
        MonitorLevel.ALL:      "    ·",
        MonitorLevel.WARNING:  " ⚠️ ",
        MonitorLevel.CRITICAL: " 🔴",
    }

    def __init__(self, level: MonitorLevel = MonitorLevel.ALL):
        self.level = level
        self.call_count = 0
        self.success_count = 0
        self.fail_count = 0
        self.rejected_count = 0
        self.in_pocket_count = 0
        self.outside_pocket_count = 0

    # ── core dispatcher ──────────────────────────────────────────────────
    def _emit(self, lvl: MonitorLevel, msg: str) -> None:
        if lvl >= self.level:
            prefix = self._PREFIXES.get(lvl, "")
            print(f"{prefix} {msg}")

    # ── convenience loggers ──────────────────────────────────────────────
    def info(self, msg: str) -> None:
        """Verbose trace (ALL level)."""
        self._emit(MonitorLevel.ALL, msg)

    def warning(self, msg: str) -> None:
        """Non-fatal issue (WARNING level)."""
        self._emit(MonitorLevel.WARNING, msg)

    def critical(self, msg: str) -> None:
        """Fatal / blocking error (CRITICAL level)."""
        self._emit(MonitorLevel.CRITICAL, msg)

    # ── EquiBind call / return ───────────────────────────────────────────
    def equibind_call(self, protein: str, ligand: str, output_dir: str,
                      seed: int, device: str) -> None:
        """Log an outgoing EquiBind invocation."""
        self.call_count += 1
        self.info(
            f"[CALL #{self.call_count}] EquiBind ← "
            f"protein={protein}, ligand={ligand}, "
            f"seed={seed}, device={device}, out={output_dir}"
        )

    def equibind_return(self, success: bool, output_sdf: Optional[str],
                        error: str = "") -> None:
        """Log what EquiBind returned."""
        if success:
            self.success_count += 1
            self.info(f"[RECV] EquiBind → OK  output={output_sdf}")
        else:
            self.fail_count += 1
            self.warning(f"[RECV] EquiBind → FAIL  error={error[:200]}")

    # ── Pocket proximity ─────────────────────────────────────────────────
    def pose_accepted_in_pocket(self, pose_id: str, pocket_id: str,
                                distance: float, threshold: float) -> None:
        """Log a pose that landed inside the target pocket."""
        self.in_pocket_count += 1
        self.info(
            f"[POCKET ✓] {pose_id} INSIDE {pocket_id}  "
            f"(dist={distance:.1f}Å ≤ {threshold:.1f}Å)"
        )

    def pose_rejected_from_pocket(self, pose_id: str, pocket_id: str,
                                  distance: float, threshold: float) -> None:
        """Log a pose that drifted outside the target pocket."""
        self.rejected_count += 1
        self.warning(
            f"[POCKET ✗] {pose_id} OUTSIDE {pocket_id}  "
            f"(dist={distance:.1f}Å > {threshold:.1f}Å) → REJECTED"
        )

    def pose_outside_all_pockets(self, pose_id: str,
                                 nearest_pocket: str,
                                 nearest_dist: float,
                                 threshold: float) -> None:
        """Log an unguided pose that is not near any known pocket."""
        self.outside_pocket_count += 1
        self.warning(
            f"[POCKET ✗] {pose_id} not in any pocket  "
            f"(nearest={nearest_pocket}, dist={nearest_dist:.1f}Å > {threshold:.1f}Å)"
        )

    def pose_inside_pocket(self, pose_id: str, pocket_id: str,
                           distance: float, threshold: float,
                           pocket_source: str) -> None:
        """Log an unguided pose that falls inside a known pocket."""
        self.in_pocket_count += 1
        self.info(
            f"[POCKET ✓] {pose_id} in {pocket_source} pocket {pocket_id}  "
            f"(dist={distance:.1f}Å ≤ {threshold:.1f}Å)"
        )

    # ── Section headers ──────────────────────────────────────────────────
    def header(self, msg: str) -> None:
        """Always printed regardless of level."""
        print(f"\n{'─' * 70}")
        print(f"  {msg}")
        print(f"{'─' * 70}")

    def section(self, msg: str) -> None:
        """Printed at ALL level."""
        self._emit(MonitorLevel.ALL, f"── {msg} ──")

    # ── Summary ──────────────────────────────────────────────────────────
    def print_summary(self) -> None:
        """Print accumulated counters (always shown)."""
        print(f"\n{'═' * 70}")
        print("  SIGNAL MONITOR SUMMARY")
        print(f"{'═' * 70}")
        print(f"  EquiBind calls:        {self.call_count}")
        print(f"  Successful returns:    {self.success_count}")
        print(f"  Failed returns:        {self.fail_count}")
        print(f"  Poses inside pocket:   {self.in_pocket_count}")
        print(f"  Poses rejected/outside:{self.rejected_count + self.outside_pocket_count}")
        print(f"    ├ guided rejected:   {self.rejected_count}")
        print(f"    └ unguided outside:  {self.outside_pocket_count}")
        print(f"{'═' * 70}")


# Instantiate the global monitor using the configured level
monitor = DockingMonitor(level=MONITOR_LEVEL)
print(f"DockingMonitor initialised  —  level = {MONITOR_LEVEL.name}")
print(f"  ALL      = full trace of every call, return, and pocket check")
print(f"  WARNING  = only rejected poses, failures, and pocket drift")
print(f"  CRITICAL = only fatal errors")

DockingMonitor initialised  —  level = ALL
  ALL      = full trace of every call, return, and pocket check
  WARNING  = only rejected poses, failures, and pocket drift
  CRITICAL = only fatal errors


# Validate GPU Support

In [6]:
import torch
import subprocess

# ============================================================================
# VALIDATE GPU SUPPORT
# ============================================================================


print("=" * 80)
print("GPU SUPPORT VALIDATION")
print("=" * 80)

# Check PyTorch CUDA availability
print("\nPyTorch CUDA Status:")
print(f"  PyTorch version: {torch.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"  CUDA version: {torch.version.cuda}")
    print(f"  Number of GPUs: {torch.cuda.device_count()}")
    print(f"  Current GPU: {torch.cuda.current_device()}")
    print(f"  GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"  GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    
    # Test GPU with a simple tensor operation
    try:
        x = torch.randn(100, 100).cuda()
        y = torch.randn(100, 100).cuda()
        z = torch.matmul(x, y)
        print(f"  ✓ GPU tensor operations working")
    except Exception as e:
        print(f"  ✗ GPU tensor test failed: {e}")
else:
    print("  ⚠️  No CUDA-capable GPU detected")
    print("  EquiBind will use CPU (slower)")

# Check nvidia-smi
print("\nNVIDIA GPU Information (nvidia-smi):")
try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True, timeout=5)
    if result.returncode == 0:
        print(result.stdout)
    else:
        print("  ⚠️  nvidia-smi not available or failed")
except FileNotFoundError:
    print("  ⚠️  nvidia-smi command not found")
except Exception as e:
    print(f"  ⚠️  Error running nvidia-smi: {e}")

# Recommendation
print("\n" + "=" * 80)
if torch.cuda.is_available():
    print("✓ GPU ACCELERATION AVAILABLE")
    print(f"  Recommended setting: EQUIBIND_DEVICE = 'cuda'")
    print(f"  Current setting: EQUIBIND_DEVICE = '{EQUIBIND_DEVICE}'")
else:
    print("⚠️  GPU NOT AVAILABLE - Using CPU")
    print(f"  Recommended setting: EQUIBIND_DEVICE = 'cpu'")
    print(f"  Current setting: EQUIBIND_DEVICE = '{EQUIBIND_DEVICE}'")
print("=" * 80)

GPU SUPPORT VALIDATION

PyTorch CUDA Status:
  PyTorch version: 2.4.0+cu121
  CUDA available: True
  CUDA version: 12.1
  Number of GPUs: 1
  Current GPU: 0
  GPU Name: NVIDIA A100-SXM4-80GB
  GPU Memory: 85.10 GB
  ✓ GPU tensor operations working

NVIDIA GPU Information (nvidia-smi):
Mon Apr  6 19:40:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 565.57.01              Driver Version: 565.57.01      CUDA Version: 12.7     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          

# Pocket Cropping

In [7]:
# ============================================================================
# PROTEIN CROPPING FOR POCKET-CONSTRAINED DOCKING
# ============================================================================
# This cell implements protein cropping to constrain EquiBind's search space
# to a specific binding pocket. By extracting only residues within a defined
# radius of the pocket center, EquiBind physically cannot predict binding
# poses outside the pocket region.
#
# Key functions:
#   - crop_protein_to_pocket(): Extract residues near pocket center
#   - get_cropped_protein_for_pocket(): Cache management for cropped proteins
#   - Offset tracking to transform coordinates back to original frame
#
# Configuration is in cell 5:
#   USE_PROTEIN_CROPPING, POCKET_CROP_RADIUS, POCKET_CROP_BUFFER
# ============================================================================

from typing import Set, Dict, Tuple, Optional, List
from pathlib import Path
import numpy as np

# Include all atoms of a residue if any atom is in range
INCLUDE_FULL_RESIDUES: bool = True

# Cache: (protein_path, pocket_unique_id) → (cropped_pdb_path, offset_vector)
_cropped_protein_cache: Dict[Tuple[str, str], Tuple[Path, np.ndarray]] = {}


def _parse_pdb_atoms(pdb_path: Path) -> List[dict]:
    """Parse ATOM/HETATM records from a PDB file.
    
    Returns list of dicts with keys:
        line, record_type, atom_num, atom_name, res_name, chain, res_num, x, y, z
    """
    atoms = []
    with open(pdb_path, 'r') as f:
        for line in f:
            if line.startswith('ATOM') or line.startswith('HETATM'):
                try:
                    atoms.append({
                        'line': line,
                        'record_type': line[0:6].strip(),
                        'atom_num': int(line[6:11]),
                        'atom_name': line[12:16].strip(),
                        'res_name': line[17:20].strip(),
                        'chain': line[21:22].strip() or 'A',
                        'res_num': int(line[22:26]),
                        'x': float(line[30:38]),
                        'y': float(line[38:46]),
                        'z': float(line[46:54]),
                    })
                except (ValueError, IndexError):
                    continue
    return atoms


def _get_residues_near_center(
    atoms: List[dict],
    center: Tuple[float, float, float],
    radius: float,
    include_full_residues: bool = True,
) -> Set[Tuple[str, int]]:
    """Find residues with at least one atom within radius of center.
    
    Returns set of (chain, res_num) tuples.
    """
    center_np = np.array(center)
    near_residues: Set[Tuple[str, int]] = set()
    
    for atom in atoms:
        coord = np.array([atom['x'], atom['y'], atom['z']])
        dist = np.linalg.norm(coord - center_np)
        if dist <= radius:
            near_residues.add((atom['chain'], atom['res_num']))
    
    return near_residues


def crop_protein_to_pocket(
    protein_pdb: Path,
    pocket_center: Tuple[float, float, float],
    output_pdb: Path,
    crop_radius: float = POCKET_CROP_RADIUS,
    buffer: float = POCKET_CROP_BUFFER,
    include_full_residues: bool = INCLUDE_FULL_RESIDUES,
) -> Tuple[bool, np.ndarray, int, int]:
    """Crop a protein PDB to residues near a pocket center.
    
    Args:
        protein_pdb: Input full protein PDB
        pocket_center: (x, y, z) pocket center coordinates
        output_pdb: Output cropped PDB path
        crop_radius: Radius around pocket center to include
        buffer: Additional buffer for context residues
        include_full_residues: If True, include all atoms of selected residues
        
    Returns:
        (success, offset_vector, n_residues_kept, n_atoms_kept)
        
        offset_vector is the translation applied to center the pocket at origin.
        To convert cropped coordinates back to original frame: coord + offset_vector
    """
    effective_radius = crop_radius + buffer
    
    # Parse full protein
    atoms = _parse_pdb_atoms(protein_pdb)
    if not atoms:
        return False, np.zeros(3), 0, 0
    
    # Find residues within range
    near_residues = _get_residues_near_center(
        atoms, pocket_center, effective_radius, include_full_residues
    )
    
    if not near_residues:
        monitor.warning(f"No residues found within {effective_radius}Å of pocket center")
        return False, np.zeros(3), 0, 0
    
    # Filter atoms to keep
    if include_full_residues:
        kept_atoms = [a for a in atoms if (a['chain'], a['res_num']) in near_residues]
    else:
        # Only keep atoms actually within radius
        center_np = np.array(pocket_center)
        kept_atoms = []
        for a in atoms:
            coord = np.array([a['x'], a['y'], a['z']])
            if np.linalg.norm(coord - center_np) <= effective_radius:
                kept_atoms.append(a)
    
    if not kept_atoms:
        return False, np.zeros(3), 0, 0
    
    # ── Deduplicate atom names within each residue ────────────────────────
    # Some PDB files have duplicate atom names (e.g. two "HN" in the same
    # residue). BioPython's PDBParser drops duplicates with a warning, but
    # EquiBind's get_receptor_inference may hard-fail ("0 defined twice").
    # Keep only the first occurrence of each atom name per (chain, res_num).
    deduped_atoms = []
    _seen_atom_names: Dict[Tuple[str, int], set] = {}   # (chain, res_num) → {atom_name, ...}
    _n_dupes = 0
    for a in kept_atoms:
        key = (a['chain'], a['res_num'])
        seen = _seen_atom_names.setdefault(key, set())
        if a['atom_name'] in seen:
            _n_dupes += 1
            continue
        seen.add(a['atom_name'])
        deduped_atoms.append(a)
    if _n_dupes:
        monitor.info(f"Dropped {_n_dupes} duplicate atom name(s) from cropped PDB")
    kept_atoms = deduped_atoms
    
    # Compute offset: we'll translate so pocket center is at origin
    # This helps EquiBind by centering the binding region
    offset = np.array(pocket_center)
    
    # Write cropped PDB with translated coordinates
    output_pdb.parent.mkdir(parents=True, exist_ok=True)
    
    with open(output_pdb, 'w', encoding='utf-8') as f:
        f.write(f"REMARK   CROPPED PROTEIN - pocket center at origin\n")
        f.write(f"REMARK   Original pocket center: {pocket_center[0]:.3f} {pocket_center[1]:.3f} {pocket_center[2]:.3f}\n")
        f.write(f"REMARK   Crop radius: {effective_radius:.1f} A\n")
        f.write(f"REMARK   Residues kept: {len(near_residues)}\n")
        f.write(f"REMARK   To restore original coords: add offset ({offset[0]:.3f}, {offset[1]:.3f}, {offset[2]:.3f})\n")
        
        atom_num = 1
        for a in kept_atoms:
            # Translate to center pocket at origin
            new_x = a['x'] - offset[0]
            new_y = a['y'] - offset[1]
            new_z = a['z'] - offset[2]
            
            # Reconstruct PDB line with new coordinates
            line = a['line']
            new_line = (
                f"{line[0:6]}"  # record type
                f"{atom_num:5d}"  # atom number (renumbered)
                f"{line[11:30]}"  # atom name, res name, chain, res num
                f"{new_x:8.3f}{new_y:8.3f}{new_z:8.3f}"  # coordinates
                f"{line[54:]}"  # rest of line (occupancy, B-factor, element)
            )
            f.write(new_line)
            atom_num += 1
        
        f.write("END\n")
    
    n_residues = len(near_residues)
    n_atoms = len(kept_atoms)
    
    return True, offset, n_residues, n_atoms


def get_cropped_protein_for_pocket(
    protein_pdb: Path,
    pocket: 'PocketInfo',  # Forward reference - defined in cell 12
    prep_dir: Path,
    crop_radius: float = POCKET_CROP_RADIUS,
) -> Tuple[Optional[Path], np.ndarray]:
    """Get or create a cropped protein PDB for a specific pocket.
    
    Uses caching to avoid re-cropping for the same protein/pocket combination.
    
    Args:
        protein_pdb: Full protein PDB path
        pocket: PocketInfo object with center coordinates
        prep_dir: Directory to store cropped PDB files
        crop_radius: Radius around pocket center
        
    Returns:
        (cropped_pdb_path, offset_vector) or (None, zeros) on failure
        
        The offset_vector should be added to docked coordinates to get
        original-frame coordinates.
    """
    cache_key = (str(protein_pdb), pocket.unique_id)
    
    if cache_key in _cropped_protein_cache:
        return _cropped_protein_cache[cache_key]
    
    # Create cropped protein
    cropped_pdb = prep_dir / f"{protein_pdb.stem}_crop_{pocket.unique_id}.pdb"
    
    success, offset, n_res, n_atoms = crop_protein_to_pocket(
        protein_pdb=protein_pdb,
        pocket_center=pocket.center,
        output_pdb=cropped_pdb,
        crop_radius=crop_radius,
    )
    
    if success:
        monitor.info(f"Cropped protein for {pocket.unique_id}: {n_res} residues, {n_atoms} atoms "
                     f"(radius={crop_radius + POCKET_CROP_BUFFER:.1f}Å)")
        _cropped_protein_cache[cache_key] = (cropped_pdb, offset)
        return cropped_pdb, offset
    else:
        monitor.warning(f"Failed to crop protein for pocket {pocket.unique_id}")
        return None, np.zeros(3)


def translate_pose_back_to_original_frame(
    sdf_path: Path,
    offset: np.ndarray,
    output_path: Path,
) -> bool:
    """Translate a docked pose back to the original protein coordinate frame.
    
    After docking against a cropped protein (centered at origin), this function
    shifts the ligand coordinates back to the original frame.
    
    Args:
        sdf_path: Input SDF from docking against cropped protein
        offset: The offset vector returned by crop_protein_to_pocket
        output_path: Output SDF in original coordinate frame
        
    Returns:
        True on success, False on failure
    """
    try:
        suppl = Chem.SDMolSupplier(str(sdf_path), removeHs=False)
        mol = next(iter(suppl), None)
        if mol is None:
            return False
        
        mol = Chem.RWMol(mol)
        conf = mol.GetConformer()
        
        for i in range(mol.GetNumAtoms()):
            pos = conf.GetAtomPosition(i)
            # Add offset to restore original coordinates
            conf.SetAtomPosition(i, (
                pos.x + offset[0],
                pos.y + offset[1],
                pos.z + offset[2],
            ))
        
        writer = Chem.SDWriter(str(output_path))
        writer.write(mol)
        writer.close()
        
        return output_path.exists() and output_path.stat().st_size > 0
    except Exception as e:
        monitor.warning(f"Failed to translate pose back to original frame: {e}")
        return False


def clamp_pose_to_pocket(
    sdf_path: Path,
    pocket_center: Tuple[float, float, float],
    max_distance: float,
    output_path: Path,
) -> Tuple[bool, float, float]:
    """Clamp a docked pose so its centroid is within max_distance of pocket center.
    
    If the pose centroid is farther than max_distance from the pocket center,
    the entire pose is translated to bring the centroid exactly to max_distance
    from the pocket center (along the line from pocket to current centroid).
    
    This ensures ALL poses stay within the defined pocket region.
    
    Args:
        sdf_path: Input SDF file
        pocket_center: Target pocket center (x, y, z)
        max_distance: Maximum allowed distance from pocket center
        output_path: Output SDF with clamped coordinates
        
    Returns:
        (success, original_distance, new_distance)
    """
    try:
        suppl = Chem.SDMolSupplier(str(sdf_path), removeHs=False)
        mol = next(iter(suppl), None)
        if mol is None:
            return False, 0.0, 0.0
        
        mol = Chem.RWMol(mol)
        conf = mol.GetConformer()
        
        # Compute current centroid
        positions = conf.GetPositions()
        centroid = positions.mean(axis=0)
        pocket_np = np.array(pocket_center)
        
        # Compute distance from pocket center
        displacement = centroid - pocket_np
        original_distance = np.linalg.norm(displacement)
        
        if original_distance <= max_distance:
            # Already within bounds, just copy
            shutil.copy2(sdf_path, output_path)
            return True, original_distance, original_distance
        
        # Compute shift needed: move centroid to exactly max_distance from pocket
        # Unit vector from pocket to centroid
        direction = displacement / original_distance
        target_centroid = pocket_np + direction * max_distance
        shift = target_centroid - centroid
        
        # Apply shift to all atoms
        for i in range(mol.GetNumAtoms()):
            pos = conf.GetAtomPosition(i)
            conf.SetAtomPosition(i, (
                pos.x + shift[0],
                pos.y + shift[1],
                pos.z + shift[2],
            ))
        
        # Write clamped SDF
        writer = Chem.SDWriter(str(output_path))
        writer.write(mol)
        writer.close()
        
        new_distance = max_distance  # By construction
        
        return output_path.exists() and output_path.stat().st_size > 0, original_distance, new_distance
        
    except Exception as e:
        monitor.warning(f"Failed to clamp pose to pocket: {e}")
        return False, 0.0, 0.0


def clear_cropped_protein_cache():
    """Clear the cropped protein cache (useful for memory management)."""
    global _cropped_protein_cache
    _cropped_protein_cache.clear()
    monitor.info("Cropped protein cache cleared")


# ── Statistics helper ─────────────────────────────────────────────────────

def estimate_search_space_reduction(
    protein_pdb: Path,
    pocket_center: Tuple[float, float, float],
    crop_radius: float = POCKET_CROP_RADIUS,
) -> Tuple[int, int, float]:
    """Estimate the search space reduction from cropping.
    
    Returns:
        (original_atoms, cropped_atoms, reduction_percent)
    """
    atoms = _parse_pdb_atoms(protein_pdb)
    original_count = len(atoms)
    
    effective_radius = crop_radius + POCKET_CROP_BUFFER
    near_residues = _get_residues_near_center(atoms, pocket_center, effective_radius)
    cropped_atoms = [a for a in atoms if (a['chain'], a['res_num']) in near_residues]
    cropped_count = len(cropped_atoms)
    
    reduction = 100.0 * (1.0 - cropped_count / original_count) if original_count > 0 else 0.0
    
    return original_count, cropped_count, reduction


print("✓ Protein cropping functions loaded")
print(f"  Default crop radius: {POCKET_CROP_RADIUS}Å (+ {POCKET_CROP_BUFFER}Å buffer)")
print(f"  Include full residues: {INCLUDE_FULL_RESIDUES}")

✓ Protein cropping functions loaded
  Default crop radius: 6.0Å (+ 3.0Å buffer)
  Include full residues: True


In [8]:
# ============================================================================
# UFF POST-DOCKING MINIMIZATION
# ============================================================================
# After EquiBind produces a docked pose, the ligand geometry may contain
# steric clashes or non-ideal bond lengths/angles. This section applies
# RDKit's Universal Force Field (UFF) to minimize the ligand in the
# context of nearby protein atoms.
#
# Strategy:
#   1. Load the docked ligand and protein structures.
#   2. Extract protein residues within UFF_PROXIMITY_RADIUS of the ligand.
#   3. Combine ligand + nearby protein into a single RDKit molecule.
#   4. Build UFF force field; constrain protein atoms (high force constant).
#   5. Minimize — only ligand atoms are free to move.
#   6. Extract minimized ligand coordinates and write to output SDF.
#
# Configuration (in Config cell above):
#   UFF_MINIMIZE, UFF_MAX_ITERS, UFF_ENERGY_TOL, UFF_PROTEIN_CONSTRAINT_WT,
#   UFF_PROXIMITY_RADIUS, UFF_ADD_HYDROGENS
# ============================================================================

from copy import deepcopy


def _extract_nearby_protein_atoms(
    protein_pdb: Path,
    ligand_coords: np.ndarray,
    radius: float = UFF_PROXIMITY_RADIUS,
) -> Optional[Chem.Mol]:
    """Extract protein residues with any atom within `radius` of any ligand atom.

    Returns an RDKit Mol of the nearby protein subset, or None on failure.
    """
    try:
        protein_mol = Chem.MolFromPDBFile(str(protein_pdb), removeHs=False, sanitize=False)
        if protein_mol is None:
            return None

        # Try sanitization; fall back to partial sanitize on failure
        try:
            Chem.SanitizeMol(protein_mol)
        except Exception:
            try:
                Chem.SanitizeMol(
                    protein_mol,
                    Chem.SanitizeFlags.SANITIZE_FINDRADICALS
                    | Chem.SanitizeFlags.SANITIZE_SETAROMATICITY
                    | Chem.SanitizeFlags.SANITIZE_SETCONJUGATION
                    | Chem.SanitizeFlags.SANITIZE_SETHYBRIDIZATION
                    | Chem.SanitizeFlags.SANITIZE_SYMMRINGS,
                )
            except Exception:
                pass

        prot_conf = protein_mol.GetConformer()
        prot_positions = prot_conf.GetPositions()

        # Find protein atoms near any ligand atom
        near_atom_indices = set()
        for lig_xyz in ligand_coords:
            dists = np.linalg.norm(prot_positions - lig_xyz, axis=1)
            near_atom_indices.update(int(i) for i in np.where(dists <= radius)[0])

        if not near_atom_indices:
            return None

        # Expand to full residues so UFF has complete residue topology
        atom_info = protein_mol.GetAtomWithIdx(0).GetPDBResidueInfo()
        if atom_info is not None:
            # Collect residue ids for nearby atoms
            nearby_residues = set()
            for idx in near_atom_indices:
                info = protein_mol.GetAtomWithIdx(int(idx)).GetPDBResidueInfo()
                if info:
                    nearby_residues.add((info.GetChainId(), info.GetResidueNumber(), info.GetInsertionCode()))
            # Include all atoms from nearby residues
            full_atom_indices = set()
            for i in range(protein_mol.GetNumAtoms()):
                info = protein_mol.GetAtomWithIdx(int(i)).GetPDBResidueInfo()
                if info and (info.GetChainId(), info.GetResidueNumber(), info.GetInsertionCode()) in nearby_residues:
                    full_atom_indices.add(int(i))
            near_atom_indices = full_atom_indices

        # Create editable molecule with only nearby atoms
        emol = Chem.RWMol(protein_mol)
        atoms_to_remove = sorted(
            set(range(protein_mol.GetNumAtoms())) - near_atom_indices, reverse=True
        )
        for idx in atoms_to_remove:
            emol.RemoveAtom(int(idx))

        nearby_mol = emol.GetMol()

        # Re-initialize ring info after atom removal (RemoveAtom invalidates it)
        try:
            Chem.FastFindRings(nearby_mol)
        except Exception:
            pass

        return nearby_mol

    except Exception as e:
        monitor.warning(f"Failed to extract nearby protein atoms: {e}")
        return None


def _uff_minimize_pose(
    docked_sdf: Path,
    protein_pdb: Path,
    output_sdf: Path,
    max_iters: int = UFF_MAX_ITERS,
    energy_tol: float = UFF_ENERGY_TOL,
    force_tol: float = UFF_FORCE_TOL,
    constraint_weight: float = UFF_PROTEIN_CONSTRAINT_WT,
    proximity_radius: float = UFF_PROXIMITY_RADIUS,
    add_hs: bool = UFF_ADD_HYDROGENS,
) -> Tuple[bool, float, float, str]:
    """Minimize a docked ligand using UFF with protein environment constraints.

    The protein atoms are held in place with harmonic position constraints
    while the ligand is free to relax.  This resolves steric clashes and
    improves bond geometries without letting the ligand drift away.

    Args:
        docked_sdf:       Path to the docked ligand SDF.
        protein_pdb:      Path to the protein PDB (full or cropped).
        output_sdf:       Where to write the minimized ligand SDF.
        max_iters:        Maximum optimization steps.
        energy_tol:       Energy convergence criterion (kcal/mol).
        force_tol:        Force convergence criterion.
        constraint_weight: Harmonic constraint weight for protein atoms.
        proximity_radius:  Include protein residues within this Å of ligand.
        add_hs:           Add hydrogens before FF setup for better UFF terms.

    Returns:
        (success, energy_before, energy_after, error_message)
    """
    try:
        # ── Load docked ligand ──
        suppl = Chem.SDMolSupplier(str(docked_sdf), removeHs=False, sanitize=False)
        lig_mol = next(iter(suppl), None)
        if lig_mol is None:
            return False, 0.0, 0.0, "Could not load docked ligand SDF"

        try:
            Chem.SanitizeMol(lig_mol)
        except Exception:
            # Attempt partial sanitization for difficult molecules
            try:
                Chem.SanitizeMol(
                    lig_mol,
                    Chem.SanitizeFlags.SANITIZE_FINDRADICALS
                    | Chem.SanitizeFlags.SANITIZE_SETAROMATICITY
                    | Chem.SanitizeFlags.SANITIZE_SETCONJUGATION
                    | Chem.SanitizeFlags.SANITIZE_SETHYBRIDIZATION
                    | Chem.SanitizeFlags.SANITIZE_SYMMRINGS,
                )
            except Exception as e:
                return False, 0.0, 0.0, f"Ligand sanitization failed: {e}"

        # Ensure ring info is initialized (required by UFF)
        try:
            Chem.FastFindRings(lig_mol)
        except Exception:
            pass

        lig_conf = lig_mol.GetConformer()
        lig_coords = lig_conf.GetPositions()
        n_lig_atoms = lig_mol.GetNumAtoms()

        # ── Extract nearby protein atoms ──
        prot_nearby = _extract_nearby_protein_atoms(protein_pdb, lig_coords, proximity_radius)
        if prot_nearby is None or prot_nearby.GetNumAtoms() == 0:
            # No protein context — minimize ligand alone
            monitor.info("UFF: No nearby protein atoms found, minimizing ligand in vacuum")
            if add_hs:
                lig_mol = Chem.AddHs(lig_mol, addCoords=True)
                n_lig_atoms_with_h = lig_mol.GetNumAtoms()
            else:
                n_lig_atoms_with_h = n_lig_atoms

            if not rdForceFieldHelpers.UFFHasAllMoleculeParams(lig_mol):
                return False, 0.0, 0.0, "UFF missing parameters for ligand atoms"

            ff = rdForceFieldHelpers.UFFGetMoleculeForceField(lig_mol)
            if ff is None:
                return False, 0.0, 0.0, "Could not create UFF force field for ligand"

            e_before = ff.CalcEnergy()
            converged = ff.Minimize(maxIts=max_iters, energyTol=energy_tol, forceTol=force_tol)
            e_after = ff.CalcEnergy()

            # Remove added Hs to match original atom count
            if add_hs:
                lig_mol = Chem.RemoveHs(lig_mol)

            writer = Chem.SDWriter(str(output_sdf))
            writer.write(lig_mol)
            writer.close()
            return True, e_before, e_after, ""

        n_prot_atoms = prot_nearby.GetNumAtoms()

        # ── Optionally add hydrogens ──
        if add_hs:
            try:
                lig_mol = Chem.AddHs(lig_mol, addCoords=True)
            except Exception:
                pass  # proceed without added Hs
            try:
                prot_nearby = Chem.AddHs(prot_nearby, addCoords=True)
            except Exception:
                pass

        n_lig_with_h = lig_mol.GetNumAtoms()
        n_prot_with_h = prot_nearby.GetNumAtoms()

        # ── Combine ligand + protein into one molecule ──
        combined = Chem.CombineMols(lig_mol, prot_nearby)

        # Verify the combined mol has a valid conformer
        if combined.GetNumConformers() == 0:
            return False, 0.0, 0.0, "Combined molecule has no conformer"

        # Initialize ring info on the combined molecule (CombineMols doesn't carry it over)
        try:
            Chem.FastFindRings(combined)
        except Exception:
            pass

        # ── Check UFF parameter coverage ──
        if not rdForceFieldHelpers.UFFHasAllMoleculeParams(combined):
            # Fallback: try ligand-only minimization
            monitor.warning("UFF missing parameters for some atoms in combined system, "
                            "falling back to ligand-only minimization")
            if rdForceFieldHelpers.UFFHasAllMoleculeParams(lig_mol):
                ff = rdForceFieldHelpers.UFFGetMoleculeForceField(lig_mol)
                if ff is not None:
                    e_before = ff.CalcEnergy()
                    ff.Minimize(maxIts=max_iters, energyTol=energy_tol, forceTol=force_tol)
                    e_after = ff.CalcEnergy()
                    if add_hs:
                        lig_mol = Chem.RemoveHs(lig_mol)
                    writer = Chem.SDWriter(str(output_sdf))
                    writer.write(lig_mol)
                    writer.close()
                    return True, e_before, e_after, "ligand-only (protein params missing)"
            return False, 0.0, 0.0, "UFF parameters missing for combined system"

        # ── Build UFF force field ──
        ff = rdForceFieldHelpers.UFFGetMoleculeForceField(
            combined,
            vdwThresh=UFF_VDW_THRESH,
        )
        if ff is None:
            return False, 0.0, 0.0, "Could not create UFF force field for combined system"

        # ── Constrain protein atoms (indices n_lig_with_h .. end) ──
        for i in range(n_lig_with_h, n_lig_with_h + n_prot_with_h):
            ff.AddFixedPoint(i)

        # ── Minimize ──
        e_before = ff.CalcEnergy()
        converged = ff.Minimize(
            maxIts=max_iters,
            energyTol=energy_tol,
            forceTol=force_tol,
        )
        e_after = ff.CalcEnergy()

        convergence_msg = "converged" if converged == 0 else f"not converged (code {converged})"

        # ── Extract minimized ligand coordinates ──
        combined_conf = combined.GetConformer()
        out_lig = deepcopy(lig_mol)
        out_conf = out_lig.GetConformer()
        for i in range(n_lig_with_h):
            pos = combined_conf.GetAtomPosition(i)
            out_conf.SetAtomPosition(i, pos)

        # Remove added Hs to match original atom count
        if add_hs:
            out_lig = Chem.RemoveHs(out_lig)

        # ── Write minimized ligand ──
        output_sdf.parent.mkdir(parents=True, exist_ok=True)
        writer = Chem.SDWriter(str(output_sdf))
        writer.write(out_lig)
        writer.close()

        monitor.info(f"UFF minimization: E {e_before:.1f} → {e_after:.1f} kcal/mol "
                     f"(ΔE={e_after - e_before:.1f}), {convergence_msg}, "
                     f"{n_lig_with_h} ligand + {n_prot_with_h} protein atoms")

        return True, e_before, e_after, convergence_msg

    except Exception as e:
        monitor.warning(f"UFF minimization failed: {e}")
        return False, 0.0, 0.0, str(e)


print("✓ UFF post-docking minimization functions loaded")
print(f"  UFF minimization enabled: {UFF_MINIMIZE}")
print(f"  Max iterations: {UFF_MAX_ITERS}")
print(f"  Protein constraint weight: {UFF_PROTEIN_CONSTRAINT_WT} kcal/mol/Å²")
print(f"  Proximity radius: {UFF_PROXIMITY_RADIUS} Å")
print(f"  Add hydrogens: {UFF_ADD_HYDROGENS}")

✓ UFF post-docking minimization functions loaded
  UFF minimization enabled: True
  Max iterations: 500
  Protein constraint weight: 100.0 kcal/mol/Å²
  Proximity radius: 8.0 Å
  Add hydrogens: True


In [9]:
import sys
import dgl
import dgl.function as fn
from copy import deepcopy
from types import SimpleNamespace

import yaml
from rdkit.Geometry import Point3D

# ── DGL compatibility shim ────────────────────────────────────────────────
# Newer DGL versions removed dgl.function.copy_edge (renamed to copy_e).
# EquiBind's model code still uses copy_edge, so we patch it back in.
if not hasattr(fn, 'copy_edge'):
    fn.copy_edge = fn.copy_e
    print("  ✓ Patched dgl.function.copy_edge → copy_e (DGL compat shim)")

# ── 0. Import EquiBind in-process ─────────────────────────────────────────
#    Add EquiBind to sys.path so we can import its modules directly,
#    avoiding subprocess + conda activation overhead per call.

  ✓ Patched dgl.function.copy_edge → copy_e (DGL compat shim)


/root/miniconda3/envs/equibind/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/root/miniconda3/envs/equibind/lib/python3.9/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


# Optimized Pocket-Guided EquiBind Docking — 3-Phase CPU/GPU Pipeline

**Architecture:** Phase 1 (CPU parallel) → Phase 2 (GPU batch) → Phase 3 (CPU parallel)

| Phase | Work | Parallelism |
|-------|------|-------------|
| **Phase 1: Preparation** | Conformer generation, ligand translation, protein cropping, receptor/ligand graph building | `ThreadPoolExecutor` across pockets and conformer seeds |
| **Phase 2: Docking** | EquiBind forward pass (graph → predicted coords) | GPU batched per receptor graph |
| **Phase 3: Post-processing** | Torsion correction, Kabsch alignment, coordinate restoration, UFF minimization, file I/O | `ThreadPoolExecutor` across poses |

Timing is tracked per-phase (prep / dock / post) for each protein-ligand complex.

In [10]:
# ============================================================================
# GLOBALLY DECOUPLED 3-PHASE EQUIBIND DOCKING PIPELINE
# ============================================================================
# Architecture: Three fully independent phases across ALL protein-ligand combos.
#
#   Phase 1 (CPU, parallel): Prepare ALL proteins × ALL ligands
#     - Protein preparation (reduce), cropped proteins per pocket
#     - Receptor graph building (cached per unique protein)
#     - Conformer generation for every ligand (parallel across seeds)
#     - Ligand translation to ALL pocket centers
#     - Ligand graph + geometry graph pre-building
#     → Output: List[DockJob] — every job ready for GPU
#
#   Phase 2 (GPU, batched): Dock ALL jobs in one go
#     - Group by receptor graph, run GPU batches
#     - All forward passes in a single GPU session
#     → Output: DockJobs with predicted_coords filled
#
#   Phase 3 (CPU, parallel): Post-process ALL docked jobs in one go
#     - Torsion correction + Kabsch alignment
#     - Coordinate restoration (for cropped proteins)
#     - Pocket enforcement / clamping
#     - UFF minimization (parallel)
#     - File output + result collection
#     → Output: List[GuidedPoseResult] + timing CSV + summary JSON
#
# Timing: Each global phase is timed. Per-combo timing is also recorded.
# ============================================================================

import sys
import dgl
import time
import csv
import json
import shutil
import subprocess
import itertools
import threading
from copy import deepcopy
from types import SimpleNamespace
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed
from dataclasses import dataclass, field
from datetime import datetime

import yaml
import numpy as np
import torch
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Geometry import Point3D

# ── 0. Import EquiBind in-process ─────────────────────────────────────────
_equibind_dir = str(EQUIBIND_DIR)
if _equibind_dir not in sys.path:
    sys.path.insert(0, _equibind_dir)

from models.equibind import EquiBind as EquiBindModel
from commons.process_mols import (
    get_receptor_inference,
    get_rec_graph,
    get_lig_graph_revised,
    get_geometry_graph,
)
from commons.geometry_utils import (
    rigid_transform_Kabsch_3D,
    get_torsions,
    get_dihedral_vonMises,
    apply_changes,
)
from commons.utils import seed_all


# ── 0a. Load EquiBind model ONCE ──────────────────────────────────────────

def _load_equibind_model(equibind_dir: Path, device: str = "cuda"):
    """Load EquiBind model and train args. Called once at startup."""
    t0 = time.time()
    ckpt_path = equibind_dir / "runs" / "flexible_self_docking" / "best_checkpoint.pt"
    train_args_path = ckpt_path.parent / "train_arguments.yaml"

    with open(train_args_path) as f:
        train_args = yaml.safe_load(f)

    train_args["model_parameters"]["noise_initial"] = 0

    args = SimpleNamespace(**train_args)
    args.checkpoint = str(ckpt_path)
    args.use_rdkit_coords = args.dataset_params.get("use_rdkit_coords", True)
    args.device = device

    dev = torch.device("cuda:0" if torch.cuda.is_available() and device == "cuda" else "cpu")
    checkpoint = torch.load(str(ckpt_path), map_location=dev)

    model = EquiBindModel(
        device=dev,
        lig_input_edge_feats_dim=15,
        rec_input_edge_feats_dim=27,
        **args.model_parameters,
    )
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(dev)
    model.eval()

    seed_all(args.seed)
    load_time = time.time() - t0

    monitor.info(f"EquiBind model loaded from {ckpt_path}")
    monitor.info(f"  Device: {dev}  |  Load time: {load_time:.2f}s")
    return model, args, dev


def _load_receptor_graph(protein_pdb: Path, args: SimpleNamespace):
    """Build the EquiBind receptor graph for a protein."""
    dp = args.dataset_params
    rec, rec_coords, c_alpha_coords, n_coords, c_coords = get_receptor_inference(str(protein_pdb))
    rec_graph = get_rec_graph(
        rec, rec_coords, c_alpha_coords, n_coords, c_coords,
        use_rec_atoms=dp["use_rec_atoms"],
        rec_radius=dp["rec_graph_radius"],
        surface_max_neighbors=dp["surface_max_neighbors"],
        surface_graph_cutoff=dp["surface_graph_cutoff"],
        surface_mesh_cutoff=dp["surface_mesh_cutoff"],
        c_alpha_max_neighbors=dp["c_alpha_max_neighbors"],
    )
    return rec_graph


# Load model once now
_eb_model, _eb_args, _eb_device = _load_equibind_model(EQUIBIND_DIR, EQUIBIND_DEVICE)

_rec_graph_cache: Dict[str, object] = {}
_rec_graph_lock = threading.Lock()


# ── Data structures ───────────────────────────────────────────────────────

@dataclass
class TimingRecord:
    """Tracks timing for a single protein-ligand combo."""
    protein: str = ""
    ligand: str = ""
    phase1_prep_s: float = 0.0
    phase2_dock_s: float = 0.0
    phase3_post_s: float = 0.0
    total_s: float = 0.0
    n_poses_attempted: int = 0
    n_poses_success: int = 0


@dataclass
class PocketInfo:
    source: str
    pocket_id: int
    unique_id: str
    score: float
    center: Tuple[float, float, float]
    radius: float
    protein_name: str
    extra: dict = field(default_factory=dict)

    def to_dict(self):
        return {
            "source": self.source, "pocket_id": self.pocket_id,
            "unique_id": self.unique_id, "score": self.score,
            "center": list(self.center), "radius": self.radius,
            "protein_name": self.protein_name, "extra": self.extra,
        }


@dataclass
class GuidedPoseResult:
    protein_name: str
    ligand_name: str
    mode: str
    pocket_id: Optional[int]
    pocket_unique_id: Optional[str]
    pose_num: int
    pocket_center: Optional[Tuple[float, float, float]]
    pose_centroid: Optional[Tuple[float, float, float]]
    sdf_path: Optional[Path]
    success: bool
    error: str = ""
    uff_energy_before: Optional[float] = None
    uff_energy_after: Optional[float] = None
    uff_minimized: bool = False
    prep_time_s: float = 0.0
    dock_time_s: float = 0.0
    post_time_s: float = 0.0

    def to_dict(self):
        return {
            "protein_name": self.protein_name, "ligand_name": self.ligand_name,
            "mode": self.mode, "pocket_id": self.pocket_id,
            "pocket_unique_id": self.pocket_unique_id, "pose_num": self.pose_num,
            "pocket_center": list(self.pocket_center) if self.pocket_center else None,
            "pose_centroid": list(self.pose_centroid) if self.pose_centroid else None,
            "sdf_path": str(self.sdf_path) if self.sdf_path else None,
            "success": self.success, "error": self.error,
            "uff_minimized": self.uff_minimized,
            "uff_energy_before": self.uff_energy_before,
            "uff_energy_after": self.uff_energy_after,
            "prep_time_s": round(self.prep_time_s, 4),
            "dock_time_s": round(self.dock_time_s, 4),
            "post_time_s": round(self.post_time_s, 4),
        }


@dataclass
class DockJob:
    """A single docking job that flows through all 3 phases."""
    job_id: str
    combo_name: str                # "ligand__protein" key
    protein_pdb: Path              # the protein PDB used for docking
    protein_key: str               # cache key for receptor graph
    prepared_protein: Path         # full (uncropped) prepared protein for UFF
    ligand_sdf: Path               # translated ligand SDF
    ligand_mol: Optional[object]   # RDKit mol (loaded)
    lig_graph: Optional[object]    # DGL ligand graph
    lig_coord: Optional[object]    # input coords tensor
    geometry_graph: Optional[object]
    seed: int
    pocket: Optional[PocketInfo]
    pocket_center: Optional[Tuple[float, float, float]]
    crop_offset: Optional[np.ndarray]
    combo_dir: Path
    final_sdf: Path
    pose_num: int
    protein_name: str
    ligand_name: str
    mode: str                      # "fpocket", "p2rank", "unguided"
    prep_time: float = 0.0
    # filled after Phase 2
    predicted_coords: Optional[object] = None
    dock_time: float = 0.0
    gpu_success: bool = False
    error: str = ""


# ── Helper functions ──────────────────────────────────────────────────────

def _sdf_centroid(sdf_path: Path) -> Optional[Tuple[float, float, float]]:
    try:
        coords = []
        with open(sdf_path) as f:
            lines = f.readlines()
        if len(lines) < 5:
            return None
        parts = lines[3].split()
        n_atoms = int(parts[0])
        for i in range(4, min(4 + n_atoms, len(lines))):
            p = lines[i].split()
            if len(p) >= 3:
                coords.append([float(p[0]), float(p[1]), float(p[2])])
        if not coords:
            return None
        return tuple(np.array(coords).mean(axis=0))
    except Exception:
        return None


def _pocket_distance(a, b):
    return float(np.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2 + (a[2]-b[2])**2))


def _run_corrections_inproc(lig, lig_coord, predicted_coords):
    """Post-process: torsion fitting + Kabsch alignment."""
    input_coords = lig_coord.detach().cpu()
    prediction = predicted_coords.detach().cpu()

    lig_input = deepcopy(lig)
    conf = lig_input.GetConformer()
    for i in range(lig_input.GetNumAtoms()):
        x, y, z = input_coords.numpy()[i]
        conf.SetAtomPosition(i, Point3D(float(x), float(y), float(z)))

    lig_equibind = deepcopy(lig)
    conf = lig_equibind.GetConformer()
    for i in range(lig_equibind.GetNumAtoms()):
        x, y, z = prediction.numpy()[i]
        conf.SetAtomPosition(i, Point3D(float(x), float(y), float(z)))

    coords_pred = lig_equibind.GetConformer().GetPositions()
    Z_pt_cloud = coords_pred
    rotable_bonds = get_torsions([lig_input])
    new_dihedrals = np.zeros(len(rotable_bonds))
    for idx_t, r in enumerate(rotable_bonds):
        new_dihedrals[idx_t] = get_dihedral_vonMises(lig_input, lig_input.GetConformer(), r, Z_pt_cloud)
    optimized_mol = apply_changes(lig_input, new_dihedrals, rotable_bonds)
    optimized_conf = optimized_mol.GetConformer()
    coords_pred_optimized = optimized_conf.GetPositions()
    R, t = rigid_transform_Kabsch_3D(coords_pred_optimized.T, coords_pred.T)
    coords_pred_optimized = (R @ coords_pred_optimized.T).T + t.squeeze()
    for i in range(optimized_mol.GetNumAtoms()):
        x, y, z = coords_pred_optimized[i]
        optimized_conf.SetAtomPosition(i, Point3D(float(x), float(y), float(z)))
    return optimized_mol


def _translate_sdf_to_pocket(sdf_path: Path, pocket_center: Tuple[float, float, float],
                              output_path: Path) -> bool:
    try:
        suppl = Chem.SDMolSupplier(str(sdf_path), removeHs=False)
        mol = next(iter(suppl), None)
        if mol is None:
            mol = Chem.MolFromMol2File(str(sdf_path), removeHs=False)
        if mol is None:
            return False
        mol = Chem.RWMol(mol)
        conf = mol.GetConformer()
        positions = conf.GetPositions()
        centroid = positions.mean(axis=0)
        shift = np.array(pocket_center) - centroid
        for i in range(mol.GetNumAtoms()):
            pos = conf.GetAtomPosition(i)
            conf.SetAtomPosition(i, (pos.x + shift[0], pos.y + shift[1], pos.z + shift[2]))
        writer = Chem.SDWriter(str(output_path))
        writer.write(mol)
        writer.close()
        return output_path.exists() and output_path.stat().st_size > 0
    except Exception as e:
        monitor.warning(f"Could not translate ligand to pocket: {e}")
        return False


# ── Pocket parsing ────────────────────────────────────────────────────────

def _parse_fpocket_pocket_centroid(pocket_pdb: Path):
    coords = []
    with open(pocket_pdb, "r") as fh:
        for line in fh:
            if line.startswith("ATOM") or line.startswith("HETATM"):
                try:
                    x, y, z = float(line[30:38]), float(line[38:46]), float(line[46:54])
                    coords.append((x, y, z))
                except (ValueError, IndexError):
                    continue
    if not coords:
        return (0.0, 0.0, 0.0), 0.0
    arr = np.array(coords)
    centroid = tuple(arr.mean(axis=0))
    radius = float(np.linalg.norm(arr - arr.mean(axis=0), axis=1).max())
    return centroid, radius


def parse_fpocket_results(fpocket_dir: Path, protein_name: str, n_top: int = N_TOP_POCKETS) -> List[PocketInfo]:
    out_dirs = sorted(fpocket_dir.glob(f"{protein_name}*_out"))
    pockets: List[PocketInfo] = []
    for out_dir in out_dirs:
        pockets_dir = out_dir / "pockets"
        if not pockets_dir.exists():
            continue
        dir_suffix = out_dir.name[len(protein_name):]
        variant = dir_suffix.replace("_out", "").strip("_") or "raw"
        info_file = list(out_dir.glob("*_info.txt"))
        scores: Dict[int, float] = {}
        if info_file:
            with open(info_file[0]) as fh:
                current_pocket = None
                for line in fh:
                    line_s = line.strip()
                    if line_s.startswith("Pocket ") and ":" in line_s:
                        try:
                            current_pocket = int(line_s.split()[1])
                        except (ValueError, IndexError):
                            current_pocket = None
                    elif current_pocket is not None and line_s.startswith("Score"):
                        try:
                            scores[current_pocket] = float(line_s.split(":")[-1].strip())
                        except ValueError:
                            pass
        pocket_pdbs = sorted(pockets_dir.glob("pocket*_atm.pdb"))
        for ppdb in pocket_pdbs:
            try:
                pocket_num = int(ppdb.stem.replace("pocket", "").replace("_atm", ""))
            except ValueError:
                continue
            centroid, radius = _parse_fpocket_pocket_centroid(ppdb)
            unique_id = f"fpocket_{variant}_p{pocket_num:03d}"
            pockets.append(PocketInfo(
                source="fpocket", pocket_id=pocket_num, unique_id=unique_id,
                score=scores.get(pocket_num, 0.0), center=centroid, radius=radius,
                protein_name=protein_name,
                extra={"pdb_file": str(ppdb), "out_dir": str(out_dir), "variant": variant},
            ))
    pockets.sort(key=lambda p: p.score, reverse=True)
    return pockets[:n_top]


def parse_p2rank_results(p2rank_dir: Path, protein_name: str, n_top: int = N_TOP_POCKETS) -> List[PocketInfo]:
    pockets: List[PocketInfo] = []
    pred_files = sorted(p2rank_dir.glob(f"{protein_name}*_predictions.csv"))
    for pred_file in pred_files:
        fname_stem = pred_file.stem
        base = fname_stem.replace("_predictions", "")
        suffix_part = base[len(protein_name):].replace(".pdb", "").strip("_")
        variant = suffix_part or "raw"
        with open(pred_file) as fh:
            reader = csv.DictReader(fh)
            for row in reader:
                try:
                    def _g(key):
                        for k, v in row.items():
                            if k.strip() == key:
                                return v.strip()
                        return None
                    rank = int(_g("rank"))
                    score = float(_g("score"))
                    prob = float(_g("probability"))
                    cx, cy, cz = float(_g("center_x")), float(_g("center_y")), float(_g("center_z"))
                    name = (_g("name") or "").strip()
                    sas = float(_g("sas_points") or 0)
                except (TypeError, ValueError):
                    continue
                radius = max(5.0, np.sqrt(sas) * 0.5)
                unique_id = f"p2rank_{variant}_p{rank:03d}"
                pockets.append(PocketInfo(
                    source="p2rank", pocket_id=rank, unique_id=unique_id,
                    score=score, center=(cx, cy, cz), radius=radius,
                    protein_name=protein_name,
                    extra={"probability": prob, "sas_points": sas,
                           "pred_file": str(pred_file), "name": name, "variant": variant},
                ))
    pockets.sort(key=lambda p: p.score, reverse=True)
    return pockets[:n_top]


# ── Conformer generation ─────────────────────────────────────────────────

def _generate_conformers_for_seed(mol_block: str, seed: int, per_seed: int,
                                   output_dir: Path) -> List[Tuple[Path, int]]:
    mol = Chem.MolFromMolBlock(mol_block, removeHs=False)
    if mol is None:
        return []
    conformer_files = []
    params = AllChem.ETKDGv3()
    params.randomSeed = seed
    params.numThreads = 0
    params.useRandomCoords = True
    params.maxIterations = 500
    try:
        cids = AllChem.EmbedMultipleConfs(mol, numConfs=per_seed, params=params)
    except Exception:
        cids = []
    for ci, cid in enumerate(cids):
        try:
            AllChem.MMFFOptimizeMolecule(mol, confId=cid, maxIters=200)
        except Exception:
            pass
        cpath = output_dir / f"seed{seed}_c{ci}.sdf"
        try:
            w = Chem.SDWriter(str(cpath))
            w.write(mol, confId=cid)
            w.close()
            if cpath.exists() and cpath.stat().st_size > 0:
                conformer_files.append((cpath, seed))
        except Exception:
            pass
    return conformer_files


def _generate_conformer_sdfs(ligand_file: Path, output_dir: Path,
                              seeds=None, per_seed=None) -> List[Tuple[Path, int]]:
    if seeds is None:
        seeds = RDKIT_SEEDS
    if per_seed is None:
        per_seed = CONFORMERS_PER_SEED
    output_dir.mkdir(parents=True, exist_ok=True)
    suffix = ligand_file.suffix.lower()
    try:
        if suffix == ".sdf":
            mol = next(iter(Chem.SDMolSupplier(str(ligand_file), removeHs=False)), None)
        elif suffix == ".mol2":
            mol = Chem.MolFromMol2File(str(ligand_file), removeHs=False)
        elif suffix == ".pdb":
            mol = Chem.MolFromPDBFile(str(ligand_file), removeHs=False)
        else:
            mol = None
    except Exception:
        mol = None
    if mol is None:
        return []
    mol = Chem.AddHs(mol)
    mol_block = Chem.MolToMolBlock(mol)
    conformer_files = []
    if PARALLEL_CONFORMER_GEN and len(seeds) > 1:
        n_workers = min(N_PARALLEL_WORKERS, len(seeds))
        with ThreadPoolExecutor(max_workers=n_workers) as executor:
            futures = {executor.submit(_generate_conformers_for_seed, mol_block, s, per_seed, output_dir): s
                       for s in seeds}
            for future in as_completed(futures):
                try:
                    conformer_files.extend(future.result())
                except Exception:
                    pass
    else:
        for seed in seeds:
            conformer_files.extend(_generate_conformers_for_seed(mol_block, seed, per_seed, output_dir))
    return conformer_files


def _ensure_ligand_sdf(ligand_file: Path, prep_dir: Path) -> Path:
    ligand_name = ligand_file.stem
    lig_sdf = prep_dir / f"{ligand_name}.sdf"
    if lig_sdf.exists():
        return lig_sdf
    suffix = ligand_file.suffix.lower()
    if suffix == ".sdf":
        shutil.copy2(ligand_file, lig_sdf)
    elif suffix in (".mol2", ".pdb"):
        loader = Chem.MolFromMol2File if suffix == ".mol2" else Chem.MolFromPDBFile
        mol = loader(str(ligand_file), removeHs=False)
        if mol:
            mol = Chem.AddHs(mol)
            AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())
            w = Chem.SDWriter(str(lig_sdf)); w.write(mol); w.close()
        else:
            shutil.copy2(ligand_file, lig_sdf)
    return lig_sdf


# ── Protein / graph helpers ──────────────────────────────────────────────

def _prepare_protein(protein_pdb: Path, prep_dir: Path) -> Path:
    prepared = prep_dir / f"{protein_pdb.stem}_protein.pdb"
    if prepared.exists():
        return prepared
    reduce_exe = shutil.which("reduce")
    if reduce_exe:
        try:
            res = subprocess.run([reduce_exe, "-Quiet", str(protein_pdb)],
                                 capture_output=True, text=True, timeout=60)
            if res.returncode == 0 and res.stdout.strip():
                prepared.write_text(res.stdout)
                return prepared
        except Exception:
            pass
    shutil.copy2(protein_pdb, prepared)
    return prepared


def _build_receptor_graph_cached(protein_pdb: Path) -> object:
    key = str(protein_pdb)
    if key not in _rec_graph_cache:
        with _rec_graph_lock:
            if key not in _rec_graph_cache:
                _rec_graph_cache[key] = _load_receptor_graph(protein_pdb, _eb_args)
    return _rec_graph_cache[key]


def _prepare_dock_job(
    conf_path: Path, seed: int, pocket: Optional[PocketInfo],
    prepared_protein: Path, protein_key: str,
    combo_dir: Path, prep_dir: Path,
    protein_name: str, ligand_name: str, lig_sdf: Path,
    mode: str, pose_num: int, final_sdf: Path,
    cropped_protein: Optional[Path], crop_offset: Optional[np.ndarray],
) -> Optional[DockJob]:
    t0 = time.time()
    combo_name = f"{ligand_name}__{protein_name}"

    if cropped_protein is not None:
        dock_protein = cropped_protein
        dock_key = str(cropped_protein)
        target_center = (0.0, 0.0, 0.0)
    else:
        dock_protein = prepared_protein
        dock_key = protein_key
        target_center = pocket.center if pocket else None

    job_id = f"{ligand_name}__{protein_name}__{mode}_{pocket.unique_id if pocket else 'unguided'}_p{pose_num:02d}"
    translated = prep_dir / f"lig_{job_id}.sdf"

    if target_center is not None:
        ok = _translate_sdf_to_pocket(conf_path, target_center, translated)
        if not ok:
            ok = _translate_sdf_to_pocket(lig_sdf, target_center, translated)
        if not ok:
            return None
        lig_file = translated
    else:
        lig_file = conf_path

    try:
        suppl = Chem.SDMolSupplier(str(lig_file), sanitize=False, removeHs=False)
        mol = next(iter(suppl), None)
        if mol is None:
            return None
        Chem.SanitizeMol(mol)
        if not mol.HasProp("_Name"):
            mol.SetProp("_Name", ligand_name)
    except Exception:
        return None

    dp = _eb_args.dataset_params
    try:
        lig_graph = get_lig_graph_revised(
            mol, ligand_name,
            max_neighbors=dp["lig_max_neighbors"],
            use_rdkit_coords=_eb_args.use_rdkit_coords,
            radius=dp["lig_graph_radius"],
        )
        lig_graph.ndata["new_x"] = lig_graph.ndata["x"]
    except Exception:
        return None

    geometry_graph = get_geometry_graph(mol) if dp.get("geometry_regularization", True) else None
    lig_coord = lig_graph.ndata["new_x"].clone()

    prep_time = time.time() - t0

    return DockJob(
        job_id=job_id,
        combo_name=combo_name,
        protein_pdb=dock_protein,
        protein_key=dock_key,
        prepared_protein=prepared_protein,
        ligand_sdf=lig_file,
        ligand_mol=mol,
        lig_graph=lig_graph,
        lig_coord=lig_coord,
        geometry_graph=geometry_graph,
        seed=seed,
        pocket=pocket,
        pocket_center=pocket.center if pocket else None,
        crop_offset=crop_offset,
        combo_dir=combo_dir,
        final_sdf=final_sdf,
        pose_num=pose_num,
        protein_name=protein_name,
        ligand_name=ligand_name,
        mode=pocket.source if pocket else "unguided",
        prep_time=prep_time,
    )


GPU_BATCH_SIZE = 8  # ligands per batch (tune based on GPU memory)


def _run_gpu_batch(jobs: List[DockJob], model, device) -> List[DockJob]:
    """Run EquiBind inference on a batch of DockJobs sharing the same receptor graph."""
    if not jobs:
        return jobs

    rec_graph = _rec_graph_cache[jobs[0].protein_key]

    t0 = time.time()
    try:
        lig_graphs = [j.lig_graph for j in jobs]
        geom_graphs = [j.geometry_graph for j in jobs]

        rec_graph_dev = rec_graph.to(device)

        for i, job in enumerate(jobs):
            lig_graphs[i] = lig_graphs[i].to(device)
            if geom_graphs[i] is not None:
                geom_graphs[i] = geom_graphs[i].to(device)

        with torch.no_grad():
            for i, job in enumerate(jobs):
                jt0 = time.time()
                try:
                    seed_all(job.seed)
                    predictions = model(lig_graphs[i], rec_graph_dev, geom_graphs[i])
                    job.predicted_coords = predictions[0][0]
                    job.gpu_success = True
                except Exception as e:
                    job.error = f"GPU inference failed: {e}"
                    job.gpu_success = False
                job.dock_time = time.time() - jt0

    except Exception as e:
        elapsed = time.time() - t0
        for job in jobs:
            job.error = f"Batch inference failed: {e}"
            job.gpu_success = False
            job.dock_time = elapsed / len(jobs)

    return jobs


def _postprocess_job(job: DockJob) -> GuidedPoseResult:
    """Post-process a single docked job: corrections, restore coords, UFF, write SDF."""
    t0 = time.time()

    if not job.gpu_success or job.predicted_coords is None:
        return GuidedPoseResult(
            protein_name=job.protein_name, ligand_name=job.ligand_name,
            mode=job.mode, pocket_id=job.pocket.pocket_id if job.pocket else None,
            pocket_unique_id=job.pocket.unique_id if job.pocket else None,
            pose_num=job.pose_num,
            pocket_center=job.pocket_center, pose_centroid=None,
            sdf_path=None, success=False, error=job.error,
            prep_time_s=job.prep_time, dock_time_s=job.dock_time,
        )

    try:
        optimized_mol = _run_corrections_inproc(job.ligand_mol, job.lig_coord, job.predicted_coords)
    except Exception as e:
        optimized_mol = deepcopy(job.ligand_mol)
        conf = optimized_mol.GetConformer()
        coords_np = job.predicted_coords.detach().cpu().numpy()
        for i in range(optimized_mol.GetNumAtoms()):
            conf.SetAtomPosition(i, Point3D(*coords_np[i].tolist()))

    pose_dir = job.combo_dir / f"{job.job_id}_run"
    pose_dir.mkdir(parents=True, exist_ok=True)
    out_sdf = pose_dir / "output.sdf"
    try:
        w = Chem.SDWriter(str(out_sdf))
        w.write(optimized_mol)
        w.close()
    except Exception as e:
        post_time = time.time() - t0
        return GuidedPoseResult(
            protein_name=job.protein_name, ligand_name=job.ligand_name,
            mode=job.mode, pocket_id=job.pocket.pocket_id if job.pocket else None,
            pocket_unique_id=job.pocket.unique_id if job.pocket else None,
            pose_num=job.pose_num,
            pocket_center=job.pocket_center, pose_centroid=None,
            sdf_path=None, success=False, error=f"SDF write failed: {e}",
            prep_time_s=job.prep_time, dock_time_s=job.dock_time, post_time_s=post_time,
        )

    if job.crop_offset is not None and np.any(job.crop_offset != 0):
        restored_sdf = pose_dir / "output_restored.sdf"
        if translate_pose_back_to_original_frame(out_sdf, job.crop_offset, restored_sdf):
            out_sdf = restored_sdf

    centroid = _sdf_centroid(out_sdf)

    if FORCE_POCKET and centroid is not None and job.pocket_center is not None:
        dist = _pocket_distance(centroid, job.pocket_center)
        if dist > POCKET_MATCH_THRESHOLD:
            if CLAMP_POSE_TO_POCKET:
                clamped_sdf = pose_dir / "output_clamped.sdf"
                clamp_ok, orig_dist, new_dist = clamp_pose_to_pocket(
                    sdf_path=out_sdf, pocket_center=job.pocket_center,
                    max_distance=POCKET_MATCH_THRESHOLD, output_path=clamped_sdf,
                )
                if clamp_ok:
                    out_sdf = clamped_sdf
                    centroid = _sdf_centroid(out_sdf)
                else:
                    post_time = time.time() - t0
                    shutil.rmtree(pose_dir, ignore_errors=True)
                    return GuidedPoseResult(
                        protein_name=job.protein_name, ligand_name=job.ligand_name,
                        mode=job.mode, pocket_id=job.pocket.pocket_id if job.pocket else None,
                        pocket_unique_id=job.pocket.unique_id if job.pocket else None,
                        pose_num=job.pose_num,
                        pocket_center=job.pocket_center, pose_centroid=None,
                        sdf_path=None, success=False,
                        error=f"Pose rejected: {dist:.1f}Å from pocket (clamping failed)",
                        prep_time_s=job.prep_time, dock_time_s=job.dock_time, post_time_s=post_time,
                    )
            else:
                post_time = time.time() - t0
                shutil.rmtree(pose_dir, ignore_errors=True)
                return GuidedPoseResult(
                    protein_name=job.protein_name, ligand_name=job.ligand_name,
                    mode=job.mode, pocket_id=job.pocket.pocket_id if job.pocket else None,
                    pocket_unique_id=job.pocket.unique_id if job.pocket else None,
                    pose_num=job.pose_num,
                    pocket_center=job.pocket_center, pose_centroid=None,
                    sdf_path=None, success=False,
                    error=f"Pose rejected: {dist:.1f}Å from pocket",
                    prep_time_s=job.prep_time, dock_time_s=job.dock_time, post_time_s=post_time,
                )

    _uff_minimized = False
    _uff_e_before = None
    _uff_e_after = None
    if UFF_MINIMIZE:
        minimized_sdf = pose_dir / "output_uff_minimized.sdf"
        uff_ok, e_before, e_after, uff_msg = _uff_minimize_pose(
            docked_sdf=out_sdf, protein_pdb=job.prepared_protein, output_sdf=minimized_sdf,
        )
        if uff_ok and minimized_sdf.exists():
            out_sdf = minimized_sdf
            _uff_minimized = True
            _uff_e_before = e_before
            _uff_e_after = e_after

    job.final_sdf.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(out_sdf, job.final_sdf)
    shutil.rmtree(pose_dir, ignore_errors=True)

    centroid = _sdf_centroid(job.final_sdf)
    post_time = time.time() - t0

    return GuidedPoseResult(
        protein_name=job.protein_name, ligand_name=job.ligand_name,
        mode=job.mode,
        pocket_id=job.pocket.pocket_id if job.pocket else None,
        pocket_unique_id=job.pocket.unique_id if job.pocket else None,
        pose_num=job.pose_num,
        pocket_center=job.pocket_center, pose_centroid=centroid,
        sdf_path=job.final_sdf, success=True,
        uff_minimized=_uff_minimized,
        uff_energy_before=_uff_e_before,
        uff_energy_after=_uff_e_after,
        prep_time_s=job.prep_time,
        dock_time_s=job.dock_time,
        post_time_s=post_time,
    )


# ============================================================================
# COLLECT INPUTS
# ============================================================================

def collect_input_files(directory: Path, extensions: List[str],
                        name_filter: str = "") -> List[Path]:
    files = []
    for ext in extensions:
        files.extend(sorted(p for p in directory.glob(f"*{ext}")
                            if p.is_file() and (not name_filter or name_filter in p.stem)))
    return sorted(set(files))


ligand_files = collect_input_files(drugs_dir, [".sdf", ".pdb", ".mol2"])
receptor_files = collect_input_files(receptors_dir, [".pdb"], name_filter=RECEPTOR_NAME_FILTER)

n_fp_total = N_TOP_POCKETS * POSES_PER_POCKET
n_pr_total = N_TOP_POCKETS * POSES_PER_POCKET
n_per_combo = n_fp_total + n_pr_total + N_UNGUIDED_POSES
total_combos = len(receptor_files) * len(ligand_files)

monitor.header("GLOBALLY DECOUPLED 3-PHASE EQUIBIND PIPELINE")
print(f"  Architecture: Phase1(ALL prep) → Phase2(ALL dock) → Phase3(ALL post)")
print(f"  Monitor level: {MONITOR_LEVEL.name}")
print(f"  EquiBind device: {_eb_device}")
print(f"  GPU batch size: {GPU_BATCH_SIZE}")
print(f"  Proteins: {len(receptor_files)}")
for p in receptor_files:
    print(f"    - {p.name}")
print(f"  Ligands: {len(ligand_files)}")
for l in ligand_files:
    print(f"    - {l.name}")
print(f"\n  Pockets: top {N_TOP_POCKETS} fpocket + {N_TOP_POCKETS} p2rank")
print(f"  Poses per pocket: {POSES_PER_POCKET}")
print(f"  Unguided poses: {N_UNGUIDED_POSES}")
print(f"  Total poses/combo: {n_per_combo}")
print(f"  Total combos: {total_combos}")
print(f"  Expected grand total: {total_combos * n_per_combo}")
print(f"\n  Pocket enforcement: {'ON' if FORCE_POCKET else 'OFF'}")
print(f"  Protein cropping: {'ON' if USE_PROTEIN_CROPPING else 'OFF'}")
print(f"  UFF minimization: {'ON' if UFF_MINIMIZE else 'OFF'}")
print(f"  Workers: {N_PARALLEL_WORKERS}")
print()

# Collect pockets per protein
protein_fpocket: Dict[str, List[PocketInfo]] = {}
protein_p2rank: Dict[str, List[PocketInfo]] = {}
for prot in receptor_files:
    pname = prot.stem
    fp = parse_fpocket_results(fpocket_results_folder, pname, N_TOP_POCKETS)
    pr = parse_p2rank_results(p2rank_folder, pname, N_TOP_POCKETS)
    protein_fpocket[pname] = fp
    protein_p2rank[pname] = pr
    monitor.info(f"{pname}: {len(fp)} fpocket + {len(pr)} p2rank pockets")

print()

# Pre-compute properties
lig_props_cache, prot_props_cache = precompute_properties(receptor_files, ligand_files)

# Init docking log
log_path, existing_combos = init_docking_log(POCKET_GUIDED_OUTPUT_DIR)
print(f"Docking log: {log_path} ({len(existing_combos)} existing entries)")

pipeline_start = time.time()


# ======================================================================
# ██████  ██   ██  █████  ███████ ███████      ██
# ██   ██ ██   ██ ██   ██ ██      ██          ███
# ██████  ███████ ███████ ███████ █████        ██
# ██      ██   ██ ██   ██      ██ ██           ██
# ██      ██   ██ ██   ██ ███████ ███████      ██
#
# CPU-PARALLEL PREPARATION — ALL COMBOS
# ======================================================================

phase1_start = time.time()
monitor.header("PHASE 1: CPU PREPARATION — ALL COMBOS")

ALL_DOCK_JOBS: List[DockJob] = []
ALL_CACHED_RESULTS: List[GuidedPoseResult] = []

# Track per-combo Phase 1 times
combo_phase1_times: Dict[str, float] = {}

# Pre-cache: prepared proteins + receptor graphs (per unique protein)
prepared_proteins: Dict[str, Path] = {}   # protein_name → prepared_pdb
protein_keys: Dict[str, str] = {}         # protein_name → cache key

for protein in receptor_files:
    pname = protein.stem
    prep_dir = POCKET_GUIDED_OUTPUT_DIR / f"_prep_{pname}"
    prep_dir.mkdir(parents=True, exist_ok=True)
    prepared = _prepare_protein(protein, prep_dir)
    prepared_proteins[pname] = prepared
    protein_keys[pname] = str(prepared)
    monitor.info(f"Building receptor graph for {prepared.name}")
    _build_receptor_graph_cached(prepared)

# Pre-cache: cropped proteins per pocket (parallel across all pockets)
all_pockets_flat: List[Tuple[str, PocketInfo]] = []
for prot in receptor_files:
    pname = prot.stem
    for p in protein_fpocket.get(pname, []) + protein_p2rank.get(pname, []):
        all_pockets_flat.append((pname, p))

cropped_map: Dict[str, Dict[str, Tuple[Optional[Path], np.ndarray]]] = defaultdict(dict)

def _prepare_cropped_for_pocket(pname, pocket):
    if not USE_PROTEIN_CROPPING:
        return pname, pocket.unique_id, None, np.zeros(3)
    prep_dir = POCKET_GUIDED_OUTPUT_DIR / f"_prep_{pname}"
    prepared = prepared_proteins[pname]
    cropped, offset = get_cropped_protein_for_pocket(
        protein_pdb=prepared, pocket=pocket,
        prep_dir=prep_dir, crop_radius=POCKET_CROP_RADIUS,
    )
    if cropped is not None:
        _build_receptor_graph_cached(cropped)
    return pname, pocket.unique_id, cropped, offset

if all_pockets_flat:
    n_workers = min(N_PARALLEL_WORKERS, len(all_pockets_flat))
    monitor.info(f"Preparing cropped proteins for {len(all_pockets_flat)} pockets ({n_workers} workers)")
    with ThreadPoolExecutor(max_workers=n_workers) as executor:
        futures = {executor.submit(_prepare_cropped_for_pocket, pn, pk): (pn, pk)
                   for pn, pk in all_pockets_flat}
        for future in as_completed(futures):
            try:
                pname, uid, cropped, offset = future.result()
                cropped_map[pname][uid] = (cropped, offset)
            except Exception as e:
                pn, pk = futures[future]
                monitor.warning(f"Cropped protein prep failed for {pn}/{pk.unique_id}: {e}")
                cropped_map[pn][pk.unique_id] = (None, np.zeros(3))

# Pre-cache: conformers per ligand (these are independent of protein)
ligand_conformers: Dict[str, List[Tuple[Path, int]]] = {}  # ligand_name → [(sdf, seed)]
ligand_sdfs: Dict[str, Path] = {}                           # ligand_name → base sdf

for ligand in ligand_files:
    lname = ligand.stem
    lig_prep_dir = POCKET_GUIDED_OUTPUT_DIR / f"_prep_lig_{lname}"
    lig_prep_dir.mkdir(parents=True, exist_ok=True)

    lig_sdf = _ensure_ligand_sdf(ligand, lig_prep_dir)
    ligand_sdfs[lname] = lig_sdf

    conf_dir = lig_prep_dir / "conformers"
    conformer_files = _generate_conformer_sdfs(ligand, conf_dir, RDKIT_SEEDS, CONFORMERS_PER_SEED)
    if not conformer_files:
        conformer_files = [(lig_sdf, 42)]
    ligand_conformers[lname] = conformer_files
    monitor.info(f"Ligand {lname}: {len(conformer_files)} conformers")

# Now build DockJobs for ALL combos
for protein, ligand in itertools.product(receptor_files, ligand_files):
    pname = protein.stem
    lname = ligand.stem
    combo_name = f"{lname}__{pname}"
    combo_t0 = time.time()

    combo_dir = POCKET_GUIDED_OUTPUT_DIR / combo_name
    combo_dir.mkdir(parents=True, exist_ok=True)
    prep_dir = combo_dir / "prep"
    prep_dir.mkdir(parents=True, exist_ok=True)

    prepared_protein = prepared_proteins[pname]
    pkey = protein_keys[pname]
    lig_sdf = ligand_sdfs[lname]
    conformer_files = ligand_conformers[lname]

    fp_pockets = protein_fpocket.get(pname, [])
    pr_pockets = protein_p2rank.get(pname, [])
    all_pockets = fp_pockets + pr_pockets

    # Check for cached results
    results_file = combo_dir / "guided_results.json"
    if SKIP_EXISTING and results_file.exists():
        try:
            cached = json.loads(results_file.read_text())
            expected = len(all_pockets) * POSES_PER_POCKET + N_UNGUIDED_POSES
            if len(cached) >= expected:
                for r in cached:
                    ALL_CACHED_RESULTS.append(GuidedPoseResult(
                        protein_name=r["protein_name"], ligand_name=r["ligand_name"],
                        mode=r["mode"], pocket_id=r.get("pocket_id"),
                        pocket_unique_id=r.get("pocket_unique_id"),
                        pose_num=r.get("pose_num", 1),
                        pocket_center=tuple(r["pocket_center"]) if r.get("pocket_center") else None,
                        pose_centroid=tuple(r["pose_centroid"]) if r.get("pose_centroid") else None,
                        sdf_path=Path(r["sdf_path"]) if r.get("sdf_path") else None,
                        success=r["success"], error=r.get("error", ""),
                    ))
                monitor.info(f"[SKIP] {combo_name}: {len(cached)} cached results")
                combo_phase1_times[combo_name] = time.time() - combo_t0
                continue
        except Exception:
            pass

    # Build jobs for pocket-guided poses
    def _build_jobs_for_pocket(pocket):
        jobs = []
        uid = pocket.unique_id
        cropped, offset = cropped_map.get(pname, {}).get(uid, (None, np.zeros(3)))
        conf_idx = 0
        for pose_num in range(1, POSES_PER_POCKET + 1):
            final_sdf = combo_dir / f"{uid}_pose{pose_num:02d}.sdf"
            if final_sdf.exists() and SKIP_EXISTING:
                centroid = _sdf_centroid(final_sdf)
                ALL_CACHED_RESULTS.append(GuidedPoseResult(
                    protein_name=pname, ligand_name=lname,
                    mode=pocket.source, pocket_id=pocket.pocket_id,
                    pocket_unique_id=pocket.unique_id, pose_num=pose_num,
                    pocket_center=pocket.center, pose_centroid=centroid,
                    sdf_path=final_sdf, success=True,
                ))
                continue
            if conf_idx >= len(conformer_files):
                break
            conf_path, seed = conformer_files[conf_idx]
            conf_idx += 1
            job = _prepare_dock_job(
                conf_path=conf_path, seed=seed, pocket=pocket,
                prepared_protein=prepared_protein, protein_key=pkey,
                combo_dir=combo_dir, prep_dir=prep_dir,
                protein_name=pname, ligand_name=lname,
                lig_sdf=lig_sdf, mode=pocket.source, pose_num=pose_num,
                final_sdf=final_sdf, cropped_protein=cropped, crop_offset=offset,
            )
            if job is not None:
                jobs.append(job)
        return jobs

    if all_pockets:
        n_workers = min(N_PARALLEL_WORKERS, len(all_pockets))
        with ThreadPoolExecutor(max_workers=n_workers) as executor:
            futures = [executor.submit(_build_jobs_for_pocket, p) for p in all_pockets]
            for future in as_completed(futures):
                try:
                    ALL_DOCK_JOBS.extend(future.result())
                except Exception as e:
                    monitor.warning(f"Job preparation failed for {combo_name}: {e}")

    # Build unguided jobs
    unguided_count = 0
    for conf_path, seed in conformer_files:
        if unguided_count >= N_UNGUIDED_POSES:
            break
        pose_num = unguided_count + 1
        final_sdf = combo_dir / f"unguided_{pose_num:03d}.sdf"
        if final_sdf.exists() and SKIP_EXISTING:
            centroid = _sdf_centroid(final_sdf)
            ALL_CACHED_RESULTS.append(GuidedPoseResult(
                protein_name=pname, ligand_name=lname,
                mode="unguided", pocket_id=None, pocket_unique_id=None,
                pose_num=pose_num, pocket_center=None, pose_centroid=centroid,
                sdf_path=final_sdf, success=True,
            ))
            unguided_count += 1
            continue
        job = _prepare_dock_job(
            conf_path=conf_path, seed=seed, pocket=None,
            prepared_protein=prepared_protein, protein_key=pkey,
            combo_dir=combo_dir, prep_dir=prep_dir,
            protein_name=pname, ligand_name=lname,
            lig_sdf=lig_sdf, mode="unguided", pose_num=pose_num,
            final_sdf=final_sdf, cropped_protein=None, crop_offset=None,
        )
        if job is not None:
            ALL_DOCK_JOBS.append(job)
            unguided_count += 1

    combo_phase1_times[combo_name] = time.time() - combo_t0

phase1_time = time.time() - phase1_start

# Phase 1 summary
n_jobs = len(ALL_DOCK_JOBS)
n_cached = len(ALL_CACHED_RESULTS)
n_unique_proteins = len(set(j.protein_key for j in ALL_DOCK_JOBS))
n_unique_ligands = len(set(j.ligand_name for j in ALL_DOCK_JOBS))

print(f"\n{'─' * 80}")
print(f"  PHASE 1 COMPLETE in {phase1_time:.1f}s ({phase1_time/60:.1f} min)")
print(f"  Jobs prepared:       {n_jobs}")
print(f"  Cached results:      {n_cached}")
print(f"  Unique proteins:     {n_unique_proteins}")
print(f"  Unique ligands:      {n_unique_ligands}")
print(f"  Receptor graphs:     {len(_rec_graph_cache)}")
print(f"{'─' * 80}")

/tmp/ipykernel_97780/2254958450.py:93: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(str(ckpt_path), map_location=dev)


[2026-04-06 19:40:24.314663] [ Using Seed :  1  ]
    · EquiBind model loaded from /workspace/EquiBind/runs/flexible_self_docking/best_checkpoint.pt
    ·   Device: cuda:0  |  Load time: 0.40s

──────────────────────────────────────────────────────────────────────
  GLOBALLY DECOUPLED 3-PHASE EQUIBIND PIPELINE
──────────────────────────────────────────────────────────────────────
  Architecture: Phase1(ALL prep) → Phase2(ALL dock) → Phase3(ALL post)
  Monitor level: ALL
  EquiBind device: cuda:0
  GPU batch size: 8
  Proteins: 4
    - Orai1WT-MDSnap-Fr300_cleaned.pdb
    - Orai1WT-MDSnap-Fr400_cleaned.pdb
    - Orai1WT-MDSnap-Fr499_cleaned.pdb
    - Orai1WT-START-Fr0_cleaned.pdb
  Ligands: 600
    - 05J_ideal.sdf
    - 07J_ideal.sdf
    - 0LI_ideal.sdf
    - 10A_ideal.sdf
    - 11U_ideal.sdf
    - 128_ideal.sdf
    - 138_ideal.sdf
    - 140_ideal.sdf
    - 146_ideal.sdf
    - 169_ideal.sdf
    - 16A_ideal.sdf
    - 17F_ideal.sdf
    - 1JY_ideal.sdf
    - 1L3_ideal.sdf
    - 1N1_ideal.s

KeyboardInterrupt: Embedding cancelled

## Phase 2: GPU Inference — All Jobs

In [ ]:
# ======================================================================
# ██████  ██   ██  █████  ███████ ███████     ██████
# ██   ██ ██   ██ ██   ██ ██      ██               ██
# ██████  ███████ ███████ ███████ █████        █████
# ██      ██   ██ ██   ██      ██ ██          ██
# ██      ██   ██ ██   ██ ███████ ███████     ███████
#
# GPU INFERENCE — ALL JOBS IN ONE GO
# ======================================================================

phase2_start = time.time()
monitor.header(f"PHASE 2: GPU INFERENCE — {n_jobs} TOTAL JOBS")

if n_jobs > 0:
    # Group ALL jobs by protein_key
    jobs_by_protein: Dict[str, List[DockJob]] = defaultdict(list)
    for job in ALL_DOCK_JOBS:
        jobs_by_protein[job.protein_key].append(job)

    total_batches = 0
    for pkey, pjobs in jobs_by_protein.items():
        prot_name = Path(pkey).name
        n_batches = (len(pjobs) + GPU_BATCH_SIZE - 1) // GPU_BATCH_SIZE
        total_batches += n_batches
        monitor.info(f"  {prot_name}: {len(pjobs)} jobs → {n_batches} batches")

        for batch_start in range(0, len(pjobs), GPU_BATCH_SIZE):
            batch = pjobs[batch_start:batch_start + GPU_BATCH_SIZE]
            _run_gpu_batch(batch, _eb_model, _eb_device)

    phase2_time = time.time() - phase2_start

    gpu_ok = sum(1 for j in ALL_DOCK_JOBS if j.gpu_success)
    gpu_fail = sum(1 for j in ALL_DOCK_JOBS if not j.gpu_success)

    print(f"\n{'─' * 80}")
    print(f"  PHASE 2 COMPLETE in {phase2_time:.1f}s ({phase2_time/60:.1f} min)")
    print(f"  Total batches:       {total_batches}")
    print(f"  Success:             {gpu_ok}")
    print(f"  Failed:              {gpu_fail}")
    if gpu_ok > 0:
        avg_dock = sum(j.dock_time for j in ALL_DOCK_JOBS if j.gpu_success) / gpu_ok
        print(f"  Avg inference time:  {avg_dock*1000:.1f}ms per ligand")
    print(f"{'─' * 80}")
else:
    phase2_time = 0.0
    print(f"\n  No jobs to dock (all cached). Phase 2 skipped.")

## Phase 3: CPU Post-Processing + Results

In [ ]:
# ======================================================================
# ██████  ██   ██  █████  ███████ ███████     ██████
# ██   ██ ██   ██ ██   ██ ██      ██               ██
# ██████  ███████ ███████ ███████ █████        █████
# ██      ██   ██ ██   ██      ██ ██               ██
# ██      ██   ██ ██   ██ ███████ ███████     ██████
#
# CPU-PARALLEL POST-PROCESSING — ALL DOCKED JOBS
# ======================================================================

phase3_start = time.time()
monitor.header(f"PHASE 3: CPU POST-PROCESSING — {n_jobs} TOTAL JOBS")

ALL_NEW_RESULTS: List[GuidedPoseResult] = []

if n_jobs > 0:
    n_workers = min(N_PARALLEL_WORKERS, n_jobs)
    monitor.info(f"Post-processing {n_jobs} jobs with {n_workers} workers")

    if n_workers > 1:
        with ThreadPoolExecutor(max_workers=n_workers) as executor:
            futures = {executor.submit(_postprocess_job, job): job for job in ALL_DOCK_JOBS}
            done_count = 0
            for future in as_completed(futures):
                done_count += 1
                try:
                    result = future.result()
                    ALL_NEW_RESULTS.append(result)
                except Exception as e:
                    job = futures[future]
                    ALL_NEW_RESULTS.append(GuidedPoseResult(
                        protein_name=job.protein_name, ligand_name=job.ligand_name,
                        mode=job.mode,
                        pocket_id=job.pocket.pocket_id if job.pocket else None,
                        pocket_unique_id=job.pocket.unique_id if job.pocket else None,
                        pose_num=job.pose_num,
                        pocket_center=job.pocket_center, pose_centroid=None,
                        sdf_path=None, success=False, error=f"Post-processing exception: {e}",
                    ))
                if done_count % 50 == 0:
                    monitor.info(f"  Post-processed {done_count}/{n_jobs}")
    else:
        for i, job in enumerate(ALL_DOCK_JOBS):
            try:
                result = _postprocess_job(job)
                ALL_NEW_RESULTS.append(result)
            except Exception as e:
                ALL_NEW_RESULTS.append(GuidedPoseResult(
                    protein_name=job.protein_name, ligand_name=job.ligand_name,
                    mode=job.mode,
                    pocket_id=job.pocket.pocket_id if job.pocket else None,
                    pocket_unique_id=job.pocket.unique_id if job.pocket else None,
                    pose_num=job.pose_num,
                    pocket_center=job.pocket_center, pose_centroid=None,
                    sdf_path=None, success=False, error=f"Post-processing exception: {e}",
                ))
            if (i + 1) % 50 == 0:
                monitor.info(f"  Post-processed {i+1}/{n_jobs}")

phase3_time = time.time() - phase3_start

n_new_ok = sum(1 for r in ALL_NEW_RESULTS if r.success)
n_new_fail = sum(1 for r in ALL_NEW_RESULTS if not r.success)

print(f"\n{'─' * 80}")
print(f"  PHASE 3 COMPLETE in {phase3_time:.1f}s ({phase3_time/60:.1f} min)")
print(f"  Success: {n_new_ok}  |  Failed: {n_new_fail}")
print(f"{'─' * 80}")


# ======================================================================
# RESULTS COLLECTION + LOGGING
# ======================================================================

pipeline_elapsed = time.time() - pipeline_start
all_guided_results = ALL_CACHED_RESULTS + ALL_NEW_RESULTS

# Save per-combo results + timing records
_timing_records: List[TimingRecord] = []

# Group results by combo
results_by_combo: Dict[str, List[GuidedPoseResult]] = defaultdict(list)
for r in all_guided_results:
    combo = f"{r.ligand_name}__{r.protein_name}"
    results_by_combo[combo].append(r)

# Also compute per-combo dock/post times from individual jobs
combo_dock_times: Dict[str, float] = defaultdict(float)
combo_post_times: Dict[str, float] = defaultdict(float)
combo_n_attempted: Dict[str, int] = defaultdict(int)

for job in ALL_DOCK_JOBS:
    combo_dock_times[job.combo_name] += job.dock_time
    combo_n_attempted[job.combo_name] += 1
for r in ALL_NEW_RESULTS:
    combo = f"{r.ligand_name}__{r.protein_name}"
    combo_post_times[combo] += r.post_time_s

for protein, ligand in itertools.product(receptor_files, ligand_files):
    pname = protein.stem
    lname = ligand.stem
    combo_name = f"{lname}__{pname}"
    combo_results = results_by_combo.get(combo_name, [])

    p1 = combo_phase1_times.get(combo_name, 0.0)
    p2 = combo_dock_times.get(combo_name, 0.0)
    p3 = combo_post_times.get(combo_name, 0.0)

    timing = TimingRecord(
        protein=pname, ligand=lname,
        phase1_prep_s=p1, phase2_dock_s=p2, phase3_post_s=p3,
        total_s=p1 + p2 + p3,
        n_poses_attempted=combo_n_attempted.get(combo_name, 0),
        n_poses_success=sum(1 for r in combo_results if r.success),
    )
    _timing_records.append(timing)

    # Save per-combo results JSON
    combo_dir = POCKET_GUIDED_OUTPUT_DIR / combo_name
    combo_dir.mkdir(parents=True, exist_ok=True)
    results_file = combo_dir / "guided_results.json"
    with open(results_file, "w") as f:
        json.dump([r.to_dict() for r in combo_results], f, indent=2)

    # Log to docking log
    n_ok = sum(1 for r in combo_results if r.success)
    n_fail = sum(1 for r in combo_results if not r.success)
    combo_status = "success" if n_ok > 0 else "failed"
    combo_error = f"{n_fail} pose(s) failed" if n_fail > 0 else ""
    combo_elapsed = p1 + p2 + p3

    lig_props = lig_props_cache.get(ligand, get_ligand_properties(ligand))
    prot_props = prot_props_cache.get(protein, get_protein_properties(protein))
    append_log_row(
        log_path=log_path, existing_combos=existing_combos,
        combo_name=combo_name, protein_name=pname, ligand_name=lname,
        status=combo_status, error_reason=combo_error,
        elapsed_time=combo_elapsed, combo_results=combo_results,
        lig_props=lig_props, prot_props=prot_props,
    )
    if combo_status == "failed":
        record_failure(POCKET_GUIDED_OUTPUT_DIR, combo_name, combo_error,
                       pname, lname, combo_elapsed)


# ── Statistics: unguided poses vs pockets ─────────────────────────────────

import pandas as pd

monitor.header("STATISTICS: UNGUIDED POSES vs POCKETS")

all_pockets_by_protein = defaultdict(lambda: {"fpocket": [], "p2rank": []})
for pname, pockets in protein_fpocket.items():
    all_pockets_by_protein[pname]["fpocket"] = pockets
for pname, pockets in protein_p2rank.items():
    all_pockets_by_protein[pname]["p2rank"] = pockets

unguided_results = [r for r in all_guided_results if r.mode == "unguided" and r.success and r.pose_centroid]
total_unguided = 0
total_in_fpocket = 0
total_in_p2rank = 0
total_in_either = 0

for r in unguided_results:
    total_unguided += 1
    fp_pockets = all_pockets_by_protein[r.protein_name]["fpocket"]
    pr_pockets = all_pockets_by_protein[r.protein_name]["p2rank"]
    in_fp = any(_pocket_distance(r.pose_centroid, p.center) <= POCKET_MATCH_THRESHOLD for p in fp_pockets)
    in_pr = any(_pocket_distance(r.pose_centroid, p.center) <= POCKET_MATCH_THRESHOLD for p in pr_pockets)
    if in_fp: total_in_fpocket += 1
    if in_pr: total_in_p2rank += 1
    if in_fp or in_pr: total_in_either += 1

    # Check if unguided landed near any pocket
    all_known_pockets = fp_pockets + pr_pockets
    for pocket in all_known_pockets:
        dist = _pocket_distance(r.pose_centroid, pocket.center)
        if dist <= POCKET_MATCH_THRESHOLD:
            monitor.pose_inside_pocket(
                pose_id=f"unguided_{r.pose_num:03d}",
                pocket_id=pocket.unique_id,
                distance=dist, threshold=POCKET_MATCH_THRESHOLD,
                pocket_source=pocket.source,
            )

print(f"  Unguided poses: {total_unguided}")
print(f"  In fpocket: {total_in_fpocket}  In p2rank: {total_in_p2rank}  In either: {total_in_either}")


# ── TIMING REPORT ─────────────────────────────────────────────────────────

monitor.header("TIMING REPORT")

print(f"\n  GLOBAL PHASE TIMING:")
print(f"  {'Phase':<25} {'Time (s)':<12} {'Time (min)':<12} {'%':>6}")
print(f"  {'─'*25} {'─'*12} {'─'*12} {'─'*6}")
total_phases = phase1_time + phase2_time + phase3_time
for label, t in [("Phase 1: CPU Prep", phase1_time),
                  ("Phase 2: GPU Dock", phase2_time),
                  ("Phase 3: CPU Post", phase3_time)]:
    pct = (t / total_phases * 100) if total_phases > 0 else 0
    print(f"  {label:<25} {t:<12.1f} {t/60:<12.1f} {pct:>5.1f}%")
print(f"  {'─'*25} {'─'*12} {'─'*12} {'─'*6}")
print(f"  {'TOTAL (phases)':<25} {total_phases:<12.1f} {total_phases/60:<12.1f} {'100.0%':>6}")
print(f"  {'Pipeline wall time':<25} {pipeline_elapsed:<12.1f} {pipeline_elapsed/60:<12.1f}")

print(f"\n\n  PER-COMBO TIMING:")
print(f"  {'Protein':<40} {'Ligand':<25} {'Prep(s)':<10} {'Dock(s)':<10} {'Post(s)':<10} {'Total(s)':<10}")
print(f"  {'─'*40} {'─'*25} {'─'*10} {'─'*10} {'─'*10} {'─'*10}")
for t in _timing_records:
    print(f"  {t.protein:<40} {t.ligand:<25} {t.phase1_prep_s:<10.1f} {t.phase2_dock_s:<10.1f} "
          f"{t.phase3_post_s:<10.1f} {t.total_s:<10.1f}")

total_prep = sum(t.phase1_prep_s for t in _timing_records)
total_dock = sum(t.phase2_dock_s for t in _timing_records)
total_post = sum(t.phase3_post_s for t in _timing_records)
total_all = sum(t.total_s for t in _timing_records)
n_records = len(_timing_records) or 1

print(f"\n  {'TOTAL':<40} {'':<25} {total_prep:<10.1f} {total_dock:<10.1f} {total_post:<10.1f} {total_all:<10.1f}")
print(f"  {'AVERAGE':<40} {'':<25} {total_prep/n_records:<10.1f} {total_dock/n_records:<10.1f} "
      f"{total_post/n_records:<10.1f} {total_all/n_records:<10.1f}")

# Per-pose timing from results
all_ok = [r for r in ALL_NEW_RESULTS if r.success and r.dock_time_s > 0]
if all_ok:
    avg_prep = np.mean([r.prep_time_s for r in all_ok])
    avg_dock = np.mean([r.dock_time_s for r in all_ok])
    avg_post = np.mean([r.post_time_s for r in all_ok])
    print(f"\n  Per-pose averages ({len(all_ok)} successful poses):")
    print(f"    Prep:  {avg_prep*1000:.1f}ms")
    print(f"    Dock:  {avg_dock*1000:.1f}ms")
    print(f"    Post:  {avg_post*1000:.1f}ms")
    print(f"    Total: {(avg_prep+avg_dock+avg_post)*1000:.1f}ms")


# ── Save timing CSV ──
timing_csv = POCKET_GUIDED_OUTPUT_DIR / "pipeline_timing.csv"
timing_df = pd.DataFrame([{
    "protein": t.protein, "ligand": t.ligand,
    "phase1_prep_s": round(t.phase1_prep_s, 2),
    "phase2_dock_s": round(t.phase2_dock_s, 2),
    "phase3_post_s": round(t.phase3_post_s, 2),
    "total_s": round(t.total_s, 2),
    "n_attempted": t.n_poses_attempted,
    "n_success": t.n_poses_success,
} for t in _timing_records])
timing_df.to_csv(timing_csv, index=False)
print(f"\n  Timing saved to: {timing_csv}")

# Save full summary
full_summary = {
    "timestamp": datetime.now().isoformat(),
    "architecture": "globally decoupled 3-phase pipeline",
    "config": {
        "gpu_batch_size": GPU_BATCH_SIZE,
        "n_top_pockets": N_TOP_POCKETS,
        "poses_per_pocket": POSES_PER_POCKET,
        "n_unguided_poses": N_UNGUIDED_POSES,
        "pocket_match_threshold_A": POCKET_MATCH_THRESHOLD,
        "force_pocket": FORCE_POCKET,
        "uff_minimize": UFF_MINIMIZE,
        "n_parallel_workers": N_PARALLEL_WORKERS,
        "use_protein_cropping": USE_PROTEIN_CROPPING,
    },
    "global_timing": {
        "pipeline_wall_time_s": round(pipeline_elapsed, 1),
        "phase1_prep_s": round(phase1_time, 1),
        "phase2_dock_s": round(phase2_time, 1),
        "phase3_post_s": round(phase3_time, 1),
    },
    "totals": {
        "combos": total_combos,
        "total_jobs_prepared": n_jobs,
        "total_cached": n_cached,
        "poses_success": n_new_ok + n_cached,
        "poses_failed": n_new_fail,
        "unguided_in_pocket": total_in_either,
    },
    "monitor_counters": {
        "equibind_calls": monitor.call_count,
        "equibind_success": monitor.success_count,
        "equibind_fail": monitor.fail_count,
    },
    "per_combo_timing": [
        {"protein": t.protein, "ligand": t.ligand,
         "prep_s": round(t.phase1_prep_s, 2),
         "dock_s": round(t.phase2_dock_s, 2),
         "post_s": round(t.phase3_post_s, 2),
         "total_s": round(t.total_s, 2)}
        for t in _timing_records
    ],
    "all_results": [r.to_dict() for r in all_guided_results],
}
summary_json = POCKET_GUIDED_OUTPUT_DIR / "pipeline_summary.json"
with open(summary_json, "w") as f:
    json.dump(full_summary, f, indent=2)
print(f"  Summary saved to: {summary_json}")

# Docking log timing
print(f"\n{'─' * 80}")
print("DOCKING LOG TIMING SUMMARY")
print(f"{'─' * 80}")
if log_path.exists():
    try:
        _log_df = pd.read_csv(log_path)
        _docked = _log_df[_log_df["status"].isin(["success", "failed"])]
        if not _docked.empty:
            print(f"  Mean time/combo: {_docked['elapsed_time_s'].mean():.1f}s")
            print(f"  Total docking:   {_docked['elapsed_time_s'].sum():.1f}s "
                  f"({_docked['elapsed_time_s'].sum()/60:.1f} min)")
    except Exception as _e:
        print(f"  Could not read log: {_e}")
print(f"  Pipeline wall time: {pipeline_elapsed:.1f}s ({pipeline_elapsed/60:.1f} min)")
print(f"{'─' * 80}")

monitor.print_summary()

# Docking Log

In [ ]:
log_path = POCKET_GUIDED_OUTPUT_DIR / DOCKING_LOG_FILENAME

if log_path.exists():
    docking_log_df = pd.read_csv(log_path)

    print("=" * 80)
    print("DOCKING LOG")
    print("=" * 80)
    print(f"\nLog file: {log_path}")
    print(f"Total entries: {len(docking_log_df)}")

    # Status breakdown
    status_counts = docking_log_df["status"].value_counts()
    print(f"\nStatus breakdown:")
    for status, count in status_counts.items():
        print(f"  {status}: {count}")

    # Pose counts
    print(f"\nPose counts:")
    for col in ["total_poses", "fpocket_poses", "p2rank_poses", "unguided_poses", "failed_poses"]:
        if col in docking_log_df.columns:
            print(f"  {col}: {int(docking_log_df[col].sum())}")

    # Show failed entries with reasons
    failed = docking_log_df[docking_log_df["status"] == "failed"]
    if not failed.empty:
        print(f"\n{'─' * 80}")
        print(f"FAILED DOCKINGS ({len(failed)}):")
        print(f"{'─' * 80}")
        for _, row in failed.iterrows():
            reason = row["error_reason"][:120] if pd.notna(row["error_reason"]) else "unknown"
            print(f"  ✗ {row['combo_name']}: {reason}")

    # Ligand property summary
    lig_cols = [c for c in ["ligand_name", "lig_molecular_weight", "lig_heavy_atoms", "lig_total_atoms",
                "lig_rotatable_bonds", "lig_num_rings", "lig_aromatic_rings",
                "lig_hbd", "lig_hba", "lig_tpsa", "lig_logp", "lig_formula"]
                if c in docking_log_df.columns]
    if lig_cols:
        lig_summary = docking_log_df[lig_cols].drop_duplicates(subset=["ligand_name"]).sort_values("ligand_name")
        print(f"\n{'─' * 80}")
        print("LIGAND PROPERTIES:")
        print(f"{'─' * 80}")
        display(lig_summary.reset_index(drop=True))

    # Protein property summary
    prot_cols = [c for c in ["protein_name", "prot_num_residues", "prot_num_atoms", "prot_num_chains"]
                 if c in docking_log_df.columns]
    if prot_cols:
        prot_summary = docking_log_df[prot_cols].drop_duplicates(subset=["protein_name"]).sort_values("protein_name")
        print(f"\n{'─' * 80}")
        print("PROTEIN PROPERTIES:")
        print(f"{'─' * 80}")
        display(prot_summary.reset_index(drop=True))

    # Timing summary
    docked = docking_log_df[docking_log_df["status"].isin(["success", "failed"])]
    if not docked.empty and "elapsed_time_s" in docked.columns:
        print(f"\n{'─' * 80}")
        print("TIMING:")
        print(f"{'─' * 80}")
        print(f"  Mean time per combination: {docked['elapsed_time_s'].mean():.1f}s")
        print(f"  Max time:  {docked['elapsed_time_s'].max():.1f}s")
        print(f"  Min time:  {docked['elapsed_time_s'].min():.1f}s")
        print(f"  Total:     {docked['elapsed_time_s'].sum():.1f}s ({docked['elapsed_time_s'].sum()/60:.1f} min)")
else:
    print(f"No docking log found at {log_path}. Run docking first.")